In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# import statsmodels.api as sm



DeepFolderPath = "/net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/"
Figs_path = DeepFolderPath + "figures/"


from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')
import os

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from adjustText import adjust_text
from sklearn.preprocessing import StandardScaler
import warnings 

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, cross_val_predict
from scipy.stats import pearsonr

In [2]:
gender = 'male'
percentile = 0.25  # 10% for top and bottom
percent = int(percentile * 100)
# age_predictions_path = DeepFolderPath + f"/prediction_results/accu_age_{gender}/cv_combined_predictions_embedding_wavLMLarge_{seed}.csv"
# age_predictions_path = DeepFolderPath + f"/Age_predictions/prediction_results/age_{gender}/combined_predictions_embedding_wavLMLarge_20.csv"
# age_predictions_path = DeepFolderPath + f"/Ridge_stuff/ridge_age_prediction_{gender}/predictions.csv"
age_predictions_path = DeepFolderPath + f"/Ridge_stuff/ridge_age_prediction_{gender}_bmi_regressed_out/predictions.csv"


subject_details = pd.read_csv(os.path.join(DeepFolderPath, 'subject_details_df_Oct25.csv'), index_col='filename')
wavLM_table = pd.read_csv(os.path.join(DeepFolderPath, 'WavLM_features.csv'), index_col=0)
classical_features = pd.read_csv(os.path.join(DeepFolderPath, 'audio_features_egemaps_good_quality.csv'), index_col=0)
classical_features_meta = pd.read_csv(os.path.join(DeepFolderPath, 'audio_features_egemaps_metadata_good_quality.csv'), index_col=0)
risk_factors_merged = pd.read_csv(os.path.join(DeepFolderPath, 'merged_risk_factors_subject_details.csv'), index_col=0)
yanir_data = "/net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/age_prediction_new_pipeline/data/"
dxa_data = pd.read_csv(yanir_data + 'X_DEXA_age.csv', index_col=[0,1]).drop(columns='RegistrationCode')

In [90]:
yanir_data = "/net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/age_prediction_new_pipeline/data/"
dxa_data = pd.read_csv(yanir_data + 'X_DEXA_age.csv', index_col=[0,1]).drop(columns='RegistrationCode')
sleep_data = pd.read_csv(yanir_data + 'X_sleep_age.csv', index_col=[0,1]).drop(columns=['RegistrationCode', 'sat_below_88', 'neurokit_hrv_frequency_ulf_during_wake'])
bt_data = pd.read_csv(yanir_data + 'X_blood_test_age.csv', index_col=[0,1]).drop(columns='RegistrationCode')
retina_data = pd.read_csv(yanir_data + 'X_retina_age.csv', index_col=[0,1]).drop(columns=['RegistrationCode', 'retina_l_eye_participant_id',
                                                                          'retina_r_eye_participant_id'])
diet_data = pd.read_csv(yanir_data + 'X_diet_age.csv', index_col=[0,1]).drop(columns=['RegistrationCode'])
lifestyle_data = pd.read_csv(yanir_data + 'X_lifestyle_age.csv', index_col=[0,1]).drop(columns='RegistrationCode')
microbiome_data = pd.read_csv(yanir_data + 'X_microbiome_age.csv', index_col=[0,1]).drop(columns='RegistrationCode')
NMR_data = pd.read_csv(yanir_data + 'X_NMR_age.csv', index_col=[0,1]).drop(columns=['RegistrationCode', 'HDL3_C', 'HDL2_C', 'Free_C', 'Esterified_C'])
metabolomics_data = pd.read_csv(yanir_data + 'X_metabolomics_age.csv', index_col=[0,1]).drop(columns=['RegistrationCode', 'bmi'])
voice_data = pd.read_csv(yanir_data + 'X_voice_age.csv', index_col=[0,1]).drop(columns=['RegistrationCode'])

dxa_age =  pd.read_csv(yanir_data + 'Y_DEXA_age.csv', index_col=[0,1])
sleep_age = pd.read_csv(yanir_data + 'Y_sleep_age.csv', index_col=[0,1])
bt_age = pd.read_csv(yanir_data + 'Y_blood_test_age.csv', index_col=[0,1])
retina_age = pd.read_csv(yanir_data + 'Y_retina_age.csv', index_col=[0,1])
diet_age = pd.read_csv(yanir_data + 'Y_diet_age.csv', index_col=[0,1])
lifestyle_age = pd.read_csv(yanir_data + 'Y_lifestyle_age.csv', index_col=[0,1])
microbiome_age = pd.read_csv(yanir_data + 'Y_microbiome_age.csv', index_col=[0,1])
NMR_age = pd.read_csv(yanir_data + 'Y_NMR_age.csv', index_col=[0,1])
metabolomics_age = pd.read_csv(yanir_data + 'Y_metabolomics_age.csv', index_col=[0,1])

In [4]:
# join features and age targets
dxa_full = dxa_data.join(dxa_age).reset_index()
sleep_full = sleep_data.join(sleep_age).reset_index()
bt_full = bt_data.join(bt_age).reset_index()
retina_full = retina_data.join(retina_age).reset_index()
diet_full = diet_data.join(diet_age).reset_index()
lifestyle_full = lifestyle_data.join(lifestyle_age).reset_index()   
microbiome_full = microbiome_data.join(microbiome_age).reset_index()
NMR_full = NMR_data.join(NMR_age).reset_index()
metabolomics_full = metabolomics_data.join(metabolomics_age).reset_index()

In [9]:
# remove from NMR_full all cols with FC or pct string in their name
NMR_clean = NMR_full[[col for col in NMR_full.columns if '_FC' not in col and '_pct' not in col]].copy()
NMR_clean = NMR_clean.dropna(subset=['gender'])
NMR_cols = [col for col in NMR_data.columns if '_FC' not in col and '_pct' not in col and ':' not in col]
NMR_cols_exta = [col for col in NMR_data.columns if ':' not in col]

In [49]:
# import pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import json
from sklearn.model_selection import GroupKFold

def ridge_groupcv_with_exports(
    df: pd.DataFrame,
    target_col: str,
    group_col: str,
    output_dir: str,
    *,
    columns: list | None = None,
    handle_nans: str = "impute",
    impute_strategy: str = "median",
    n_splits: int = 5,
    random_state: int = 42,
    alpha: float = 1.0,
    standardize: bool = True,             # control scaling
    shap_sample: int = 2000,
    save_plots: bool = True,
    split_gender: bool = False,           # split by gender if requested
    split_gender_post_train: bool = False, # NEW: train on all, evaluate by gender
    optimize_alpha: bool = False,         # optimize alpha on validation set
    alpha_candidates: list | None = None, # list of alpha values to try
    validation_fraction: float = 0.2,     # fraction of training for validation
    regress_out_confounds: bool = False,  # NEW: regress out confounding variables
    confound_columns: list | None = None, # NEW: list of confounding variable columns
    confound_alpha: float = 1.0           # NEW: alpha for confound regression
):
    """
    Ridge regression with GroupKFold CV and optional scaling.
    Optionally splits by gender (0 and 1) and runs separate analyses.
    Optionally optimizes alpha using a validation set in each fold.
    Optionally regresses out confounding variables from input features.
    Exports predictions, metrics, coefficients, and SHAP analysis.
    
    Parameters
    ----------
    optimize_alpha : bool
        If True, optimize alpha on a validation subset of training data in each fold.
    alpha_candidates : list | None
        List of alpha values to try during optimization. 
        If None, uses [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0].
    validation_fraction : float
        Fraction of training groups to use as validation set for alpha optimization.
    split_gender_post_train : bool
        If True, train on all data but evaluate separately for each gender in test sets.
        Requires 'gender' column in df.
    regress_out_confounds : bool
        If True, regress out confounding variables from input features using ridge regression.
    confound_columns : list | None
        List of column names to use as confounding variables. Required if regress_out_confounds=True.
    confound_alpha : float
        Alpha parameter for ridge regression when regressing out confounds.
    """
    os.makedirs(output_dir, exist_ok=True)

    if target_col not in df.columns:
        raise ValueError(f"target_col '{target_col}' not found")
    if group_col not in df.columns:
        raise ValueError(f"group_col '{group_col}' not found")
    
    # Validate confound parameters
    if regress_out_confounds:
        if confound_columns is None or len(confound_columns) == 0:
            raise ValueError("confound_columns must be provided when regress_out_confounds=True")
        missing_confounds = [c for c in confound_columns if c not in df.columns]
        if missing_confounds:
            raise ValueError(f"Confound columns not found in df: {missing_confounds}")
    
    # Validate split_gender_post_train
    if split_gender_post_train:
        if "gender" not in df.columns:
            raise ValueError("split_gender_post_train=True requires 'gender' column in df")
        if split_gender:
            raise ValueError("split_gender and split_gender_post_train cannot both be True")

    # Handle optional gender splitting as a wrapper
    if split_gender and "gender" in df.columns:
        metrics_rows = []
        results_by_gender = {}

        for gender_value, label in [(1, "male"), (0, "female")]:
            sub_df = df[df["gender"] == gender_value].copy()
            if sub_df.empty:
                print(f"No rows found for gender == {gender_value}, skipping")
                continue

            sub_out_dir = os.path.join(output_dir, f"gender_{label}")
            metrics = ridge_groupcv_with_exports(
                df=sub_df,
                target_col=target_col,
                group_col=group_col,
                output_dir=sub_out_dir,
                columns=columns,
                handle_nans=handle_nans,
                impute_strategy=impute_strategy,
                n_splits=n_splits,
                random_state=random_state,
                alpha=alpha,
                standardize=standardize,
                shap_sample=shap_sample,
                save_plots=save_plots,
                split_gender=False,  # avoid recursion splitting again
                split_gender_post_train=False,
                optimize_alpha=optimize_alpha,
                alpha_candidates=alpha_candidates,
                validation_fraction=validation_fraction,
                regress_out_confounds=regress_out_confounds,
                confound_columns=confound_columns,
                confound_alpha=confound_alpha
            )

            row = {"gender": label}
            row.update(metrics)
            metrics_rows.append(row)
            results_by_gender[label] = metrics

        if metrics_rows:
            metrics_df = pd.DataFrame(metrics_rows)
            metrics_df.to_csv(
                os.path.join(output_dir, "metrics_by_gender.csv"),
                index=False
            )
            
            # Also save as JSON for easier programmatic access
            with open(os.path.join(output_dir, "metrics_by_gender.json"), "w") as f:
                json.dump(results_by_gender, f, indent=2)
            
            print(f"Saved gender specific metrics in {output_dir}")
            print(f"  - metrics_by_gender.csv: tabular format")
            print(f"  - metrics_by_gender.json: structured format")
            print(f"  - gender_male/: male-specific outputs")
            print(f"  - gender_female/: female-specific outputs")
        else:
            print("split_gender=True but no gender subsets were created")

        return results_by_gender

    # If we reach here, either split_gender is False or 'gender' is not present:
    # run the original single dataset analysis.

    X_all = df.drop(columns=[target_col])
    if columns is None:
        feature_cols = X_all.select_dtypes(include=[np.number]).columns.tolist()
    else:
        missing = [c for c in columns if c not in X_all.columns]
        if missing:
            raise ValueError(f"Requested columns not in df: {missing}")
        feature_cols = columns
    if not feature_cols:
        raise ValueError("No numeric feature columns found or selected")

    # Build a single aligned dataframe for X, y, groups
    used_cols = feature_cols + [target_col, group_col]
    
    # Add confound columns to used_cols if needed
    if regress_out_confounds:
        for conf_col in confound_columns:
            if conf_col not in used_cols:
                used_cols.append(conf_col)
    
    # Add gender column if split_gender_post_train
    if split_gender_post_train:
        if "gender" not in used_cols:
            used_cols.append("gender")
    
    df_used = df[used_cols].copy()

    # Always drop rows where target or group are NaN
    df_used = df_used.dropna(subset=[target_col, group_col])

    X = df_used[feature_cols]
    y = df_used[target_col]
    groups = df_used[group_col]
    
    # Extract gender if needed
    if split_gender_post_train:
        gender_series = df_used["gender"]
    else:
        gender_series = None
    
    # Extract confound variables if needed
    if regress_out_confounds:
        confounds = df_used[confound_columns]
    else:
        confounds = None

    if handle_nans not in {"impute", "drop"}:
        raise ValueError("handle_nans must be 'impute' or 'drop'")

    if handle_nans == "drop":
        # Drop rows with NaNs in any feature, keep X, y, groups aligned
        mask = ~X.isna().any(axis=1)
        if regress_out_confounds:
            # Also check confounds for NaNs
            mask = mask & ~confounds.isna().any(axis=1)
        if split_gender_post_train:
            # Also check gender for NaNs
            mask = mask & ~gender_series.isna()
        X = X.loc[mask]
        y = y.loc[mask]
        groups = groups.loc[mask]
        if regress_out_confounds:
            confounds = confounds.loc[mask]
        if split_gender_post_train:
            gender_series = gender_series.loc[mask]
    
    if regress_out_confounds:
        print(f"Regressing out confounds: {confound_columns}")
        
        # Build preprocessing pipeline for confounds
        confound_prep_steps = []
        if handle_nans == "impute":
            confound_prep_steps.append(("imputer", SimpleImputer(strategy=impute_strategy)))
        confound_prep_steps.append(("scaler", StandardScaler()))
        confound_prep = Pipeline(confound_prep_steps)
        
        # Build preprocessing pipeline for features
        feature_prep_steps = []        
        feature_prep = Pipeline(feature_prep_steps)
        
        # Preprocess confounds and features
        confounds_processed = confound_prep.fit_transform(confounds)
        X_processed = feature_prep.fit_transform(X)
        
        # For each feature, regress it on confounds and keep residuals
        ridge_confound = Ridge(alpha=confound_alpha, random_state=random_state)
        X_residuals = np.zeros_like(X_processed)
        
        # print for debugging
        print(f"Regressing out confounds for {X_processed.shape[1]} features")
        for i in range(X_processed.shape[1]):
            y_feature = X_processed[:, i]
            ridge_confound.fit(confounds_processed, y_feature)
            y_pred = ridge_confound.predict(confounds_processed)
            X_residuals[:, i] = y_feature - y_pred
        
        # Replace X with residuals (convert back to DataFrame to preserve index)
        X = pd.DataFrame(X_residuals, index=X.index, columns=feature_cols)
        
        # Save confound regression info
        confound_info = {
            "confound_columns": confound_columns,
            "confound_alpha": confound_alpha,
            "n_confounds": len(confound_columns)
        }
        with open(os.path.join(output_dir, "confound_regression_info.json"), "w") as f:
            json.dump(confound_info, f, indent=2)
        
        print(f"Confounds regressed out. Features are now residuals.")

    # Build pipeline dynamically
    # Note: If confounds were regressed out, features are already preprocessed (imputed and scaled)
    # so we skip those steps and only apply Ridge
    steps = []
    if not regress_out_confounds:
        # Only add preprocessing if confounds were NOT regressed out
        if handle_nans == "impute":
            steps.append(("imputer", SimpleImputer(strategy=impute_strategy)))
        if standardize:
            steps.append(("scaler", StandardScaler()))
    steps.append(("ridge", Ridge(alpha=alpha, random_state=random_state)))
    pipe = Pipeline(steps)

    gkf = GroupKFold(n_splits=n_splits)

    oof = pd.Series(index=y.index, dtype=float)
    fold_r2 = []
    fold_best_alphas = []  # store best alpha per fold
    
    # For split_gender_post_train: store gender-specific metrics per fold
    if split_gender_post_train:
        fold_metrics_by_gender = {"male": [], "female": []}

    # Default alpha candidates for optimization
    if alpha_candidates is None:
        alpha_candidates = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0]

    for fold_idx, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        
        # Optionally optimize alpha using a validation split from training data
        if optimize_alpha:
            groups_tr = groups.iloc[tr_idx]
            unique_groups = groups_tr.unique()
            n_val_groups = max(1, int(len(unique_groups) * validation_fraction))
            
            np.random.seed(random_state + fold_idx)
            val_groups = np.random.choice(unique_groups, size=n_val_groups, replace=False)
            
            # Split training into train_inner and val_inner
            mask_val = groups_tr.isin(val_groups)
            X_tr_inner = X_tr.loc[~mask_val]
            y_tr_inner = y_tr.loc[~mask_val]
            X_val_inner = X_tr.loc[mask_val]
            y_val_inner = y_tr.loc[mask_val]
            
            # Try different alphas
            best_alpha = alpha
            best_val_r2 = -np.inf
            
            for alpha_candidate in alpha_candidates:
                steps_temp = []
                if not regress_out_confounds:
                    # Only add preprocessing if confounds were NOT regressed out
                    if handle_nans == "impute":
                        steps_temp.append(("imputer", SimpleImputer(strategy=impute_strategy)))
                    if standardize:
                        steps_temp.append(("scaler", StandardScaler()))
                steps_temp.append(("ridge", Ridge(alpha=alpha_candidate, random_state=random_state)))
                pipe_temp = Pipeline(steps_temp)
                
                pipe_temp.fit(X_tr_inner, y_tr_inner)
                y_pred_val_inner = pipe_temp.predict(X_val_inner)
                val_r2 = r2_score(y_val_inner, y_pred_val_inner)
                
                if val_r2 > best_val_r2:
                    best_val_r2 = val_r2
                    best_alpha = alpha_candidate
            
            fold_best_alphas.append(best_alpha)
            current_alpha = best_alpha
            print(f"Fold {fold_idx}: Best alpha = {best_alpha:.4f} (Val R2 = {best_val_r2:.4f})")
        else:
            current_alpha = alpha
        
        # Build pipeline with optimized or fixed alpha
        steps_fold = []
        if not regress_out_confounds:
            # Only add preprocessing if confounds were NOT regressed out
            if handle_nans == "impute":
                steps_fold.append(("imputer", SimpleImputer(strategy=impute_strategy)))
            if standardize:
                steps_fold.append(("scaler", StandardScaler()))
        steps_fold.append(("ridge", Ridge(alpha=current_alpha, random_state=random_state)))
        pipe_fold = Pipeline(steps_fold)
        
        pipe_fold.fit(X_tr, y_tr)
        y_pred = pipe_fold.predict(X_va)
        oof.iloc[va_idx] = y_pred
        fold_r2.append(r2_score(y_va, y_pred))
        
        # If split_gender_post_train, compute gender-specific metrics for this fold
        if split_gender_post_train:
            gender_va = gender_series.iloc[va_idx]
            for gender_value, label in [(1, "male"), (0, "female")]:
                gender_mask = gender_va == gender_value
                if gender_mask.sum() > 0:
                    y_va_gender = y_va[gender_mask]
                    y_pred_gender = y_pred[gender_mask]
                    
                    gender_r2 = r2_score(y_va_gender, y_pred_gender)
                    gender_r = pearsonr(y_va_gender, y_pred_gender)[0] if np.std(y_pred_gender) > 0 else np.nan
                    gender_mae = mean_absolute_error(y_va_gender, y_pred_gender)
                    gender_rmse = mean_squared_error(y_va_gender, y_pred_gender)
                    
                    fold_metrics_by_gender[label].append({
                        "fold": fold_idx,
                        "n_samples": int(gender_mask.sum()),
                        "R2": float(gender_r2),
                        "Pearson_r": float(gender_r),
                        "MAE": float(gender_mae),
                        "RMSE": float(gender_rmse)
                    })

    oof_r2 = r2_score(y, oof)
    oof_r = pearsonr(y, oof)[0] if np.std(oof) > 0 else np.nan
    oof_mae = mean_absolute_error(y, oof)
    oof_rmse = mean_squared_error(y, oof, squared=False)

    # Fit on all data (use average of best alphas if optimized, else use fixed alpha)
    if optimize_alpha and fold_best_alphas:
        final_alpha = np.mean(fold_best_alphas)
        print(f"Final model alpha (average of fold bests): {final_alpha:.4f}")
    else:
        final_alpha = alpha
    
    steps_final = []
    if not regress_out_confounds:
        # Only add preprocessing if confounds were NOT regressed out
        if handle_nans == "impute":
            steps_final.append(("imputer", SimpleImputer(strategy=impute_strategy)))
        if standardize:
            steps_final.append(("scaler", StandardScaler()))
    steps_final.append(("ridge", Ridge(alpha=final_alpha, random_state=random_state)))
    pipe = Pipeline(steps_final)
    
    pipe.fit(X, y)
    ridge = pipe.named_steps["ridge"]
    coefs = pd.Series(ridge.coef_.ravel(), index=feature_cols, name="coef")
    coef_df = pd.DataFrame({"feature": coefs.index, "coef": coefs.values})
    coef_df["abs_coef"] = coef_df["coef"].abs()
    coef_df.sort_values("abs_coef", ascending=False).to_csv(
        os.path.join(output_dir, "coefficients.csv"), index=False
    )

    # Save predictions and fold assignments
    pred_df = pd.DataFrame({
        "index": oof.index.astype(str),
        "group": groups.astype(str),
        "true_values": y.values,
        "predictions": oof.values,
    })
    fold_marks = pd.Series(index=y.index, dtype="Int64")
    for fold_idx, (_, va_idx) in enumerate(gkf.split(X, y, groups), start=1):
        fold_marks.iloc[va_idx] = fold_idx
    pred_df["fold"] = fold_marks.values
    
    # Add gender column if split_gender_post_train
    if split_gender_post_train:
        pred_df["gender"] = gender_series.values
    
    pred_df.to_csv(os.path.join(output_dir, "predictions.csv"), index=False)

    # Compute and save gender-specific overall metrics if split_gender_post_train
    if split_gender_post_train:
        gender_metrics_overall = {}
        
        for gender_value, label in [(1, "male"), (0, "female")]:
            gender_mask = gender_series == gender_value
            if gender_mask.sum() > 0:
                y_gender = y[gender_mask]
                oof_gender = oof[gender_mask]
                
                gender_r2 = r2_score(y_gender, oof_gender)
                gender_r = pearsonr(y_gender, oof_gender)[0] if np.std(oof_gender) > 0 else np.nan
                gender_mae = mean_absolute_error(y_gender, oof_gender)
                gender_rmse = mean_squared_error(y_gender, oof_gender, squared=False)
                
                gender_metrics_overall[label] = {
                    "n_samples": int(gender_mask.sum()),
                    "oof_Pearson_r": float(gender_r),
                    "oof_R2": float(gender_r2),
                    "oof_MAE": float(gender_mae),
                    "oof_RMSE": float(gender_rmse),
                    "per_fold_metrics": fold_metrics_by_gender[label]
                }
        
        # Save gender-specific metrics
        with open(os.path.join(output_dir, "metrics_by_gender.json"), "w") as f:
            json.dump(gender_metrics_overall, f, indent=2)
        
        # Also save a summary CSV
        gender_summary = []
        for label, metrics in gender_metrics_overall.items():
            row = {"gender": label}
            row.update({k: v for k, v in metrics.items() if k != "per_fold_metrics"})
            gender_summary.append(row)
        
        if gender_summary:
            pd.DataFrame(gender_summary).to_csv(
                os.path.join(output_dir, "metrics_by_gender_summary.csv"),
                index=False
            )
        
        print(f"Saved gender-specific evaluation metrics")

    metrics = {
        "oof_Pearson_r": float(oof_r),
        "oof_R2": float(oof_r2),
        "oof_MAE": float(oof_mae),
        "oof_RMSE": float(oof_rmse),
        "per_fold_R2": [float(x) for x in fold_r2],
        "n_samples": int(X.shape[0]),
        "n_features": int(X.shape[1]),
        "alpha": float(final_alpha),
        "alpha_optimized": bool(optimize_alpha),
        "fold_best_alphas": [float(x) for x in fold_best_alphas] if optimize_alpha else None,
        "standardize": bool(standardize),
        "handle_nans": handle_nans,
        "impute_strategy": impute_strategy,
        "split_gender_post_train": bool(split_gender_post_train),
        "regress_out_confounds": bool(regress_out_confounds),
        "confound_columns": confound_columns if regress_out_confounds else None,
        "confound_alpha": float(confound_alpha) if regress_out_confounds else None
    }
    with open(os.path.join(output_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    # SHAP analysis
    try:
        import shap
        if shap_sample and X.shape[0] > shap_sample:
            bg_idx = np.random.RandomState(123).choice(
                X.index, shap_sample, replace=False
            )
            X_bg_raw = X.loc[bg_idx]
        else:
            X_bg_raw = X

        preproc = Pipeline(pipe.steps[:-1])
        X_bg = preproc.transform(X_bg_raw)
        X_full = preproc.transform(X)

        explainer = shap.LinearExplainer(ridge, X_bg, feature_names=feature_cols)
        shap_values = explainer(X_full)

        # Optional: save values as npz
        # np.savez_compressed(
        #     os.path.join(output_dir, "shap_values.npz"),
        #     values=shap_values.values,
        #     base_values=shap_values.base_values,
        #     data=X_full,
        #     feature_names=np.array(feature_cols, dtype=object)
        # )

        if save_plots:
            plt.figure(figsize=(10, 8))
            shap.summary_plot(
                shap_values.values,
                features=X_full,
                feature_names=feature_cols,
                show=False
            )
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, "shap_summary.png"), dpi=200)
            plt.close()

    except Exception as e:
        with open(os.path.join(output_dir, "shap_error.txt"), "w") as f:
            f.write(str(e))

    print(f"Saved outputs in {output_dir}")
    return metrics


In [40]:
ridge_groupcv_with_exports(
    df=NMR_full,
    target_col='age',
    group_col='subject_number',
    output_dir=DeepFolderPath + 'Ridge_stuff/ridge_NMR_age_prediction',
    columns=NMR_cols,
    handle_nans='impute',
    impute_strategy='median',
    n_splits=5,
    random_state=42,
    alpha=0.2,      
    standardize=True,
    shap_sample=2000,
    save_plots=True,
    split_gender=True
)

Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/ridge_NMR_age_prediction/gender_male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/ridge_NMR_age_prediction/gender_female
Saved gender specific metrics in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/ridge_NMR_age_prediction
  - metrics_by_gender.csv: tabular format
  - metrics_by_gender.json: structured format
  - gender_male/: male-specific outputs
  - gender_female/: female-specific outputs
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/ridge_NMR_age_prediction/gender_female
Saved gender specific metrics in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/ridge_NMR_age_prediction
  - metrics_by_gender.csv: tabular format
  - metrics_by_gender.json: s

{'male': {'oof_Pearson_r': 0.4398118760627547,
  'oof_R2': 0.19098933900867043,
  'oof_MAE': 5.728489574057288,
  'oof_RMSE': 7.052347107076452,
  'per_fold_R2': [0.23560122198167832,
   0.17963017143854976,
   0.18421833746023197,
   0.21687182570663222,
   0.12613466711843768],
  'n_samples': 4780,
  'n_features': 150,
  'alpha': 0.2,
  'alpha_optimized': False,
  'fold_best_alphas': None,
  'standardize': True,
  'handle_nans': 'impute',
  'impute_strategy': 'median',
  'split_gender_post_train': False,
  'regress_out_confounds': False,
  'confound_columns': None,
  'confound_alpha': None},
 'female': {'oof_Pearson_r': 0.5072584719780553,
  'oof_R2': 0.25629682456349345,
  'oof_MAE': 5.407582388513631,
  'oof_RMSE': 6.715980297928967,
  'per_fold_R2': [0.28104137813015195,
   0.2488135164121288,
   0.23809347834089134,
   0.27855981039428623,
   0.23100586430397219],
  'n_samples': 5285,
  'n_features': 150,
  'alpha': 0.2,
  'alpha_optimized': False,
  'fold_best_alphas': None,
  '

In [51]:
ridge_groupcv_with_exports(
    df=metabolomics_full,
    target_col='age',
    group_col='subject_number',
    output_dir=DeepFolderPath + 'Ridge_stuff/ridge_Metabo_age_prediction',
    columns=metabolomics_data.columns.to_list(),
    handle_nans='impute',
    impute_strategy='median',
    n_splits=5,
    random_state=42,
    alpha=0.2,      
    standardize=True,
    shap_sample=2000,
    save_plots=True,
    split_gender=True
)

Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/ridge_Metabo_age_prediction/gender_male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/ridge_Metabo_age_prediction/gender_female
Saved gender specific metrics in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/ridge_Metabo_age_prediction
  - metrics_by_gender.csv: tabular format
  - metrics_by_gender.json: structured format
  - gender_male/: male-specific outputs
  - gender_female/: female-specific outputs


{'male': {'oof_Pearson_r': 0.48324321589942965,
  'oof_R2': -0.4481887194304741,
  'oof_MAE': 5.884052482256954,
  'oof_RMSE': 9.224215827565212,
  'per_fold_R2': [-0.5058385050756364,
   -0.6874811556802103,
   -0.2980587006675317,
   -0.32145132446013625,
   -0.4426973473713969],
  'n_samples': 5342,
  'n_features': 2064,
  'alpha': 0.2,
  'alpha_optimized': False,
  'fold_best_alphas': None,
  'standardize': True,
  'handle_nans': 'impute',
  'impute_strategy': 'median',
  'split_gender_post_train': False,
  'regress_out_confounds': False,
  'confound_columns': None,
  'confound_alpha': None},
 'female': {'oof_Pearson_r': 0.49762591861068556,
  'oof_R2': -0.4970539489012542,
  'oof_MAE': 5.499020369389671,
  'oof_RMSE': 9.417854231971402,
  'per_fold_R2': [-0.5288665611909822,
   -0.3567468591777996,
   -1.3008670495319778,
   -0.1085561372248387,
   -0.20995964005656398],
  'n_samples': 5655,
  'n_features': 2064,
  'alpha': 0.2,
  'alpha_optimized': False,
  'fold_best_alphas': No

In [62]:
# Filter microbiome data to keep only top 10% most prevalent species
microbiome_prevalence = (microbiome_data > 0.0001).sum() / len(microbiome_data)
microbiome_cols_filtered = microbiome_prevalence[microbiome_prevalence >= 0.1].index.tolist()

print(f"Original microbiome features: {len(microbiome_data.columns)}")
print(f"Filtered microbiome features (top 10% prevalence): {len(microbiome_cols_filtered)}")

# Use filtered columns for subsequent analyses
microbiome_data_filtered = microbiome_data[microbiome_cols_filtered]

Original microbiome features: 3595
Filtered microbiome features (top 10% prevalence): 514


In [63]:
microbiome_full_filtered = microbiome_data_filtered.join(microbiome_age)

In [68]:
microbiome_full_filtered.reset_index(inplace=True)

In [71]:
# Loop over sleep, diet, microbiome, and retina datasets and run ridge regression
datasets = {
    'sleep': (sleep_full, sleep_data.columns.tolist()),
    'diet': (diet_full, diet_data.columns.tolist()),
    'microbiome': (microbiome_full_filtered, microbiome_data_filtered.columns.tolist()),
    'retina': (retina_full, retina_data.columns.tolist()),
    'NMR': (NMR_full, NMR_cols),
    'DEXA': (dxa_full, dxa_data.columns.tolist()),
    'blood_test': (bt_full, bt_data.columns.tolist()),
    'diet': (diet_full, diet_data.columns.tolist()),
}

for dataset_name, (df, feature_cols) in datasets.items():
    print(f"\n{'='*60}")
    print(f"Running Ridge Regression for {dataset_name.upper()}")
    print(f"{'='*60}")
    
    ridge_groupcv_with_exports(
        df=df,
        target_col='age',
        group_col='subject_number',
        output_dir=DeepFolderPath + f'Ridge_stuff/ridge_{dataset_name}_age_prediction',
        columns=feature_cols,
        handle_nans='impute',
        impute_strategy='median',
        n_splits=5,
        random_state=42,
        alpha=0.2,
        standardize=True,
        shap_sample=2000,
        save_plots=True,
        split_gender=True
    )
    
    print(f"Completed {dataset_name} - Results saved")



Running Ridge Regression for SLEEP
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/ridge_sleep_age_prediction/gender_male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/ridge_sleep_age_prediction/gender_male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/ridge_sleep_age_prediction/gender_female
Saved gender specific metrics in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/ridge_sleep_age_prediction
  - metrics_by_gender.csv: tabular format
  - metrics_by_gender.json: structured format
  - gender_male/: male-specific outputs
  - gender_female/: female-specific outputs
Completed sleep - Results saved

Running Ridge Regression for DIET
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/rid

In [47]:
import os
import pandas as pd
import numpy as np
import json
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr

def run_multi_seed_ridge(
    df: pd.DataFrame,
    target_col: str,
    group_col: str,
    output_dir: str,
    seeds: list[int] = [42, 1, 2, 3, 4],
    **kwargs
):
    """
    Wrapper to run ridge_groupcv_with_exports multiple times with different seeds.
    It aggregates predictions (bagging) to reduce variance and calculates 
    metrics on the averaged predictions.

    Parameters
    ----------
    df, target_col, group_col, output_dir : Same as original function.
    seeds : list[int]
        List of random seeds to run.
    **kwargs : 
        All other arguments to pass to ridge_groupcv_with_exports 
        (e.g. columns, alpha, split_gender_post_train, split_gender, etc.)
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Check if split_gender is enabled
    split_gender = kwargs.get('split_gender', False)
    
    print(f"Starting Multi-Seed Run with seeds: {seeds}")
    print("="*60)

    # Handle split_gender separately
    if split_gender:
        print("split_gender=True: Processing each gender separately across seeds")
        
        # Process each gender separately
        gender_results = {}
        for gender_value, gender_label in [(1, "male"), (0, "female")]:
            print(f"\n{'='*60}")
            print(f"Processing gender: {gender_label}")
            print(f"{'='*60}")
            
            all_pred_dfs = []
            all_coef_dfs = []
            seed_metrics_list = []
            
            # Loop through seeds for this gender
            for seed in seeds:
                print(f"--> Running Seed: {seed} for {gender_label}")
                
                # Create a subdirectory for this seed
                seed_dir = os.path.join(output_dir, f"seed_{seed}")
                
                # Update random_state in kwargs
                run_kwargs = kwargs.copy()
                run_kwargs['random_state'] = seed
                
                # Run the original function
                _ = ridge_groupcv_with_exports(
                    df=df,
                    target_col=target_col,
                    group_col=group_col,
                    output_dir=seed_dir,
                    **run_kwargs
                )
                
                # Read back the predictions for this seed and gender
                gender_pred_path = os.path.join(seed_dir, f"gender_{gender_label}", "predictions.csv")
                if os.path.exists(gender_pred_path):
                    seed_pred_df = pd.read_csv(gender_pred_path)
                    
                    # Ensure index is treated as string for consistent merging
                    seed_pred_df['index'] = seed_pred_df['index'].astype(str)
                    seed_pred_df = seed_pred_df.set_index('index')
                    
                    # Keep only relevant columns from the first seed
                    if len(all_pred_dfs) == 0:
                        base_columns = [c for c in seed_pred_df.columns if c != 'predictions' and c != 'fold']
                        base_df = seed_pred_df[base_columns].copy()
                    
                    # Extract just the prediction for this seed
                    pred_series = seed_pred_df['predictions'].rename(f"pred_seed_{seed}")
                    all_pred_dfs.append(pred_series)
                    
                    # Calculate metrics for this seed
                    y_true_seed = seed_pred_df['true_values']
                    y_pred_seed = seed_pred_df['predictions']
                    mask_seed = ~y_true_seed.isna() & ~y_pred_seed.isna()
                    y_true_seed = y_true_seed[mask_seed]
                    y_pred_seed = y_pred_seed[mask_seed]
                    
                    seed_r2 = r2_score(y_true_seed, y_pred_seed)
                    seed_r = pearsonr(y_true_seed, y_pred_seed)[0] if np.std(y_pred_seed) > 0 else np.nan
                    seed_mae = mean_absolute_error(y_true_seed, y_pred_seed)
                    seed_rmse = mean_squared_error(y_true_seed, y_pred_seed, squared=False)
                    
                    seed_metrics_list.append({
                        'seed': seed,
                        'R2': float(seed_r2),
                        'Pearson_r': float(seed_r),
                        'MAE': float(seed_mae),
                        'RMSE': float(seed_rmse)
                    })
                
                # Read back the coefficients for this seed and gender
                gender_coef_path = os.path.join(seed_dir, f"gender_{gender_label}", "coefficients.csv")
                if os.path.exists(gender_coef_path):
                    seed_coef_df = pd.read_csv(gender_coef_path)
                    seed_coef_df['seed'] = seed
                    all_coef_dfs.append(seed_coef_df)
            
            if not all_pred_dfs:
                print(f"Warning: No predictions found for {gender_label}")
                continue
            
            print(f"Aggregating results for {gender_label}...")
            
            # Aggregate Predictions (Averaging)
            preds_concat = pd.concat(all_pred_dfs, axis=1)
            
            # Calculate the mean prediction across rows (seeds)
            base_df['mean_predictions'] = preds_concat.mean(axis=1)
            
            # Determine std dev of predictions across seeds (uncertainty measure)
            base_df['pred_std'] = preds_concat.std(axis=1)
            
            # Merge the individual seed predictions back in for reference
            final_pred_df = pd.concat([base_df, preds_concat], axis=1)
            
            # Save averaged predictions for this gender
            gender_out_dir = os.path.join(output_dir, f"gender_{gender_label}")
            os.makedirs(gender_out_dir, exist_ok=True)
            final_pred_path = os.path.join(gender_out_dir, "predictions_averaged.csv")
            final_pred_df.reset_index().to_csv(final_pred_path, index=False)
            
            # Recalculate Metrics on the Averaged Prediction
            y_true = final_pred_df['true_values']
            y_pred = final_pred_df['mean_predictions']
            
            # Drop NaNs if any
            mask = ~y_true.isna() & ~y_pred.isna()
            y_true = y_true[mask]
            y_pred = y_pred[mask]

            r2 = r2_score(y_true, y_pred)
            pearson_r = pearsonr(y_true, y_pred)[0] if np.std(y_pred) > 0 else np.nan
            mae = mean_absolute_error(y_true, y_pred)
            rmse = mean_squared_error(y_true, y_pred, squared=False)
            
            # Calculate statistics across seeds
            seed_metrics_df = pd.DataFrame(seed_metrics_list)
            
            gender_metrics = {
                "gender": gender_label,
                "n_seeds": len(seeds),
                "seeds_used": seeds,
                "averaged_Pearson_r": float(pearson_r),
                "averaged_R2": float(r2),
                "averaged_MAE": float(mae),
                "averaged_RMSE": float(rmse),
                "n_samples": int(len(y_true)),
                # Per-seed statistics
                "per_seed_R2_mean": float(seed_metrics_df['R2'].mean()),
                "per_seed_R2_std": float(seed_metrics_df['R2'].std()),
                "per_seed_Pearson_r_mean": float(seed_metrics_df['Pearson_r'].mean()),
                "per_seed_Pearson_r_std": float(seed_metrics_df['Pearson_r'].std()),
                "per_seed_MAE_mean": float(seed_metrics_df['MAE'].mean()),
                "per_seed_MAE_std": float(seed_metrics_df['MAE'].std()),
                "per_seed_RMSE_mean": float(seed_metrics_df['RMSE'].mean()),
                "per_seed_RMSE_std": float(seed_metrics_df['RMSE'].std()),
                "per_seed_metrics": seed_metrics_list
            }
            
            gender_results[gender_label] = gender_metrics
            
            # Save gender-specific metrics JSON
            with open(os.path.join(gender_out_dir, "metrics_averaged.json"), "w") as f:
                json.dump(gender_metrics, f, indent=2)
            
            # Save per-seed metrics as CSV
            seed_metrics_df.to_csv(os.path.join(gender_out_dir, "metrics_per_seed.csv"), index=False)
            
            # Aggregate Coefficients for this gender
            if all_coef_dfs:
                all_coefs = pd.concat(all_coef_dfs, axis=0)
                coef_summary = all_coefs.groupby("feature")["coef"].agg(['mean', 'std', 'count']).reset_index()
                coef_summary['abs_mean_coef'] = coef_summary['mean'].abs()
                coef_summary = coef_summary.sort_values("abs_mean_coef", ascending=False)
                
                coef_summary.to_csv(os.path.join(gender_out_dir, "coefficients_averaged.csv"), index=False)
            
            print(f"Done with {gender_label}. Averaged R2: {r2:.4f} (Per-seed mean: {gender_metrics['per_seed_R2_mean']:.4f} ± {gender_metrics['per_seed_R2_std']:.4f})")
        
        # Save combined gender metrics
        if gender_results:
            gender_df = pd.DataFrame(gender_results).T
            gender_df.to_csv(os.path.join(output_dir, "metrics_averaged_by_gender.csv"))
            
            with open(os.path.join(output_dir, "metrics_averaged_by_gender.json"), "w") as f:
                json.dump(gender_results, f, indent=2)
            
            print(f"\n{'='*60}")
            print(f"All gender-specific results saved in {output_dir}")
            print(f"  - gender_male/: male-specific averaged outputs")
            print(f"  - gender_female/: female-specific averaged outputs")
            print(f"  - metrics_averaged_by_gender.csv: combined summary")
            print(f"  - metrics_averaged_by_gender.json: structured format")
        
        return gender_results
    
    # Original logic for non-split_gender case
    all_pred_dfs = []
    all_coef_dfs = []
    seed_metrics_list = []

    # 1. Loop through seeds and run the base pipeline
    for seed in seeds:
        print(f"--> Running Seed: {seed}")
        
        # Create a subdirectory for this seed to avoid file collisions
        seed_dir = os.path.join(output_dir, f"seed_{seed}")
        
        # Update random_state in kwargs
        run_kwargs = kwargs.copy()
        run_kwargs['random_state'] = seed
        
        # Run the original function
        # We ignore the return metrics here, as we will recalculate them on the average
        _ = ridge_groupcv_with_exports(
            df=df,
            target_col=target_col,
            group_col=group_col,
            output_dir=seed_dir,
            **run_kwargs
        )
        
        # Read back the predictions for this seed
        pred_path = os.path.join(seed_dir, "predictions.csv")
        if os.path.exists(pred_path):
            # Read, set index to align rows, rename prediction column
            seed_pred_df = pd.read_csv(pred_path)
            
            # Ensure index is treated as string for consistent merging
            seed_pred_df['index'] = seed_pred_df['index'].astype(str)
            seed_pred_df = seed_pred_df.set_index('index')
            
            # Keep only relevant columns from the first seed, otherwise just the prediction
            if len(all_pred_dfs) == 0:
                base_columns = [c for c in seed_pred_df.columns if c != 'predictions' and c != 'fold']
                base_df = seed_pred_df[base_columns].copy()
            
            # Extract just the prediction for this seed
            pred_series = seed_pred_df['predictions'].rename(f"pred_seed_{seed}")
            all_pred_dfs.append(pred_series)
            
            # Calculate metrics for this seed
            y_true_seed = seed_pred_df['true_values']
            y_pred_seed = seed_pred_df['predictions']
            mask_seed = ~y_true_seed.isna() & ~y_pred_seed.isna()
            y_true_seed = y_true_seed[mask_seed]
            y_pred_seed = y_pred_seed[mask_seed]
            
            seed_r2 = r2_score(y_true_seed, y_pred_seed)
            seed_r = pearsonr(y_true_seed, y_pred_seed)[0] if np.std(y_pred_seed) > 0 else np.nan
            seed_mae = mean_absolute_error(y_true_seed, y_pred_seed)
            seed_rmse = mean_squared_error(y_true_seed, y_pred_seed, squared=False)
            
            seed_metrics_list.append({
                'seed': seed,
                'R2': float(seed_r2),
                'Pearson_r': float(seed_r),
                'MAE': float(seed_mae),
                'RMSE': float(seed_rmse)
            })
        
        # Read back the coefficients for this seed
        coef_path = os.path.join(seed_dir, "coefficients.csv")
        if os.path.exists(coef_path):
            seed_coef_df = pd.read_csv(coef_path)
            seed_coef_df['seed'] = seed
            all_coef_dfs.append(seed_coef_df)

    print("="*60)
    print("Aggregating results...")

    # 2. Aggregate Predictions (Averaging)
    # Concatenate all prediction series along columns, aligning by index
    preds_concat = pd.concat(all_pred_dfs, axis=1)
    
    # Calculate the mean prediction across rows (seeds)
    base_df['mean_predictions'] = preds_concat.mean(axis=1)
    
    # Determine std dev of predictions across seeds (uncertainty measure)
    base_df['pred_std'] = preds_concat.std(axis=1)
    
    # Merge the individual seed predictions back in for reference (optional)
    final_pred_df = pd.concat([base_df, preds_concat], axis=1)
    
    # Save averaged predictions
    final_pred_path = os.path.join(output_dir, "predictions_averaged.csv")
    final_pred_df.reset_index().to_csv(final_pred_path, index=False)
    
    # 3. Recalculate Metrics on the Averaged Prediction
    y_true = final_pred_df['true_values']
    y_pred = final_pred_df['mean_predictions']
    
    # Drop NaNs if any (though pipeline should have handled them)
    mask = ~y_true.isna() & ~y_pred.isna()
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    r2 = r2_score(y_true, y_pred)
    pearson_r = pearsonr(y_true, y_pred)[0] if np.std(y_pred) > 0 else np.nan
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    
    # Calculate statistics across seeds
    seed_metrics_df = pd.DataFrame(seed_metrics_list)
    
    metrics = {
        "n_seeds": len(seeds),
        "seeds_used": seeds,
        "averaged_Pearson_r": float(pearson_r),
        "averaged_R2": float(r2),
        "averaged_MAE": float(mae),
        "averaged_RMSE": float(rmse),
        # Per-seed statistics
        "per_seed_R2_mean": float(seed_metrics_df['R2'].mean()),
        "per_seed_R2_std": float(seed_metrics_df['R2'].std()),
        "per_seed_Pearson_r_mean": float(seed_metrics_df['Pearson_r'].mean()),
        "per_seed_Pearson_r_std": float(seed_metrics_df['Pearson_r'].std()),
        "per_seed_MAE_mean": float(seed_metrics_df['MAE'].mean()),
        "per_seed_MAE_std": float(seed_metrics_df['MAE'].std()),
        "per_seed_RMSE_mean": float(seed_metrics_df['RMSE'].mean()),
        "per_seed_RMSE_std": float(seed_metrics_df['RMSE'].std()),
        "per_seed_metrics": seed_metrics_list
    }

    # 4. Handle Gender Splitting Metrics (if applicable)
    split_gender_post = kwargs.get('split_gender_post_train', False)
    if split_gender_post and 'gender' in final_pred_df.columns:
        gender_metrics = {}
        for gender_val in final_pred_df['gender'].unique():
            # Convert numpy/pandas types to native python if needed for labeling
            # Assuming gender is 0/1 or M/F strings
            label = str(gender_val)
            if label == "1" or label == "1.0": label = "male"
            if label == "0" or label == "0.0": label = "female"
            
            sub_df = final_pred_df[final_pred_df['gender'] == gender_val]
            if len(sub_df) > 0:
                y_g = sub_df['true_values']
                p_g = sub_df['mean_predictions']
                
                g_r2 = r2_score(y_g, p_g)
                g_r = pearsonr(y_g, p_g)[0] if np.std(p_g) > 0 else np.nan
                g_mae = mean_absolute_error(y_g, p_g)
                g_rmse = mean_squared_error(y_g, p_g, squared=False)
                
                gender_metrics[label] = {
                    "R2": float(g_r2),
                    "Pearson_r": float(g_r),
                    "MAE": float(g_mae),
                    "RMSE": float(g_rmse),
                    "n_samples": int(len(sub_df))
                }
        
        metrics['gender_metrics'] = gender_metrics
        
        # Save gender summary CSV
        pd.DataFrame(gender_metrics).T.to_csv(
            os.path.join(output_dir, "metrics_averaged_by_gender.csv")
        )

    # Save final metrics JSON
    with open(os.path.join(output_dir, "metrics_averaged.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    
    # Save per-seed metrics as CSV
    seed_metrics_df.to_csv(os.path.join(output_dir, "metrics_per_seed.csv"), index=False)

    # 5. Aggregate Coefficients
    if all_coef_dfs:
        all_coefs = pd.concat(all_coef_dfs, axis=0)
        # Group by feature name and calculate statistics
        coef_summary = all_coefs.groupby("feature")["coef"].agg(['mean', 'std', 'count']).reset_index()
        coef_summary['abs_mean_coef'] = coef_summary['mean'].abs()
        coef_summary = coef_summary.sort_values("abs_mean_coef", ascending=False)
        
        coef_summary.to_csv(os.path.join(output_dir, "coefficients_averaged.csv"), index=False)

    print(f"Done. Averaged results saved to {output_dir}")
    print(f"Final Averaged R2: {r2:.4f}")
    print(f"Per-seed R2: {metrics['per_seed_R2_mean']:.4f} ± {metrics['per_seed_R2_std']:.4f}")
    
    return metrics


In [48]:
# run multi-seed ridge
run_multi_seed_ridge(
    df=NMR_full,
    target_col='age',
    group_col='subject_number',
    output_dir=DeepFolderPath + 'Ridge_stuff/multi_seed_ridge_NMR_age_prediction',
    seeds=[42, 1, 2, 3, 4, 17, 99, 123, 256, 512],
    columns=NMR_cols,
    handle_nans='impute',
    impute_strategy='median',
    n_splits=5,
    alpha=0.2,      
    standardize=True,
    shap_sample=2000,
    save_plots=True,
    split_gender=True
)

Starting Multi-Seed Run with seeds: [42, 1, 2, 3, 4, 17, 99, 123, 256, 512]
split_gender=True: Processing each gender separately across seeds

Processing gender: male
--> Running Seed: 42 for male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/multi_seed_ridge_NMR_age_prediction/seed_42/gender_male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/multi_seed_ridge_NMR_age_prediction/seed_42/gender_male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/multi_seed_ridge_NMR_age_prediction/seed_42/gender_female
Saved gender specific metrics in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/multi_seed_ridge_NMR_age_prediction/seed_42
  - metrics_by_gender.csv: tabular format
  - metrics_by_gender.json: structured format
  - gender_male/: male-specific outpu

Starting Multi-Seed Run with seeds: [42, 1, 2, 3, 4, 17, 99, 123, 256, 512]
split_gender=True: Processing each gender separately across seeds

Processing gender: male
--> Running Seed: 42 for male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/multi_seed_ridge_NMR_age_prediction/seed_42/gender_male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/multi_seed_ridge_NMR_age_prediction/seed_42/gender_male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/multi_seed_ridge_NMR_age_prediction/seed_42/gender_female
Saved gender specific metrics in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/multi_seed_ridge_NMR_age_prediction/seed_42
  - metrics_by_gender.csv: tabular format
  - metrics_by_gender.json: structured format
  - gender_male/: male-specific outpu

{'male': {'gender': 'male',
  'n_seeds': 10,
  'seeds_used': [42, 1, 2, 3, 4, 17, 99, 123, 256, 512],
  'averaged_Pearson_r': 0.42570770132264063,
  'averaged_R2': 0.18120170104682376,
  'averaged_MAE': 5.800079091070136,
  'averaged_RMSE': 7.094879488650122,
  'n_samples': 4780,
  'per_seed_R2_mean': 0.1812017010468238,
  'per_seed_R2_std': 2.925694557147251e-17,
  'per_seed_Pearson_r_mean': 0.42570770132264074,
  'per_seed_Pearson_r_std': 5.851389114294502e-17,
  'per_seed_MAE_mean': 5.8000790910701365,
  'per_seed_MAE_std': 9.362222582871203e-16,
  'per_seed_RMSE_mean': 7.094879488650122,
  'per_seed_RMSE_std': 0.0,
  'per_seed_metrics': [{'seed': 42,
    'R2': 0.18120170104682376,
    'Pearson_r': 0.42570770132264074,
    'MAE': 5.800079091070136,
    'RMSE': 7.094879488650122},
   {'seed': 1,
    'R2': 0.18120170104682376,
    'Pearson_r': 0.42570770132264074,
    'MAE': 5.800079091070136,
    'RMSE': 7.094879488650122},
   {'seed': 2,
    'R2': 0.18120170104682376,
    'Pearson_r

In [183]:
import lightgbm as lgb
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from scipy.stats import uniform, randint

def lightgbm_groupcv_with_exports(
    df: pd.DataFrame,
    target_col: str,
    group_col: str,
    output_dir: str,
    *,
    columns: list | None = None,
    handle_nans: str = "impute",
    impute_strategy: str = "median",
    n_splits: int = 5,
    random_state: int = 42,
    lgbm_params: dict | None = None,
    num_boost_round: int = 100,
    early_stopping_rounds: int | None = None,
    shap_sample: int = 2000,
    save_plots: bool = True,
    split_gender: bool = False,
    split_gender_post_train: bool = False,
    regress_out_confounds: bool = False,
    confound_columns: list | None = None,
    confound_alpha: float = 1.0,
    optimize_hyperparams: bool = False,
    n_iter_search: int = 30,
    validation_fraction: float = 0.2
):
    """
    LightGBM regression with GroupKFold CV.
    Optionally splits by gender (0 and 1) and runs separate analyses.
    Exports predictions, metrics, feature importance, and SHAP analysis.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe
    target_col : str
        Target column name
    group_col : str
        Grouping column for GroupKFold
    output_dir : str
        Output directory for results
    columns : list | None
        Feature columns to use. If None, uses all numeric columns
    handle_nans : str
        'impute' or 'drop'
    impute_strategy : str
        Strategy for imputation ('median', 'mean', etc.)
    n_splits : int
        Number of CV folds
    random_state : int
        Random seed
    lgbm_params : dict | None
        LightGBM parameters. If None, uses defaults. Ignored if optimize_hyperparams=True
    num_boost_round : int
        Number of boosting rounds
    early_stopping_rounds : int | None
        Early stopping rounds (if None, no early stopping)
    shap_sample : int
        Sample size for SHAP analysis
    save_plots : bool
        Whether to save plots
    split_gender : bool
        Split analysis by gender
    split_gender_post_train : bool
        Train on all data but evaluate by gender
    regress_out_confounds : bool
        Regress out confounding variables from features
    confound_columns : list | None
        Confounding variable columns
    confound_alpha : float
        Alpha for confound regression
    optimize_hyperparams : bool
        If True, optimize hyperparameters using RandomizedSearchCV on validation set
    n_iter_search : int
        Number of parameter settings sampled in randomized search
    validation_fraction : float
        Fraction of training groups to use as validation set for hyperparameter optimization
    """
    os.makedirs(output_dir, exist_ok=True)

    if target_col not in df.columns:
        raise ValueError(f"target_col '{target_col}' not found")
    if group_col not in df.columns:
        raise ValueError(f"group_col '{group_col}' not found")
    
    # Validate confound parameters
    if regress_out_confounds:
        if confound_columns is None or len(confound_columns) == 0:
            raise ValueError("confound_columns must be provided when regress_out_confounds=True")
        missing_confounds = [c for c in confound_columns if c not in df.columns]
        if missing_confounds:
            raise ValueError(f"Confound columns not found in df: {missing_confounds}")
    
    # Validate split_gender_post_train
    if split_gender_post_train:
        if "gender" not in df.columns:
            raise ValueError("split_gender_post_train=True requires 'gender' column in df")
        if split_gender:
            raise ValueError("split_gender and split_gender_post_train cannot both be True")

    # Handle optional gender splitting as a wrapper
    if split_gender and "gender" in df.columns:
        metrics_rows = []
        results_by_gender = {}

        for gender_value, label in [(1, "male"), (0, "female")]:
            sub_df = df[df["gender"] == gender_value].copy()
            if sub_df.empty:
                print(f"No rows found for gender == {gender_value}, skipping")
                continue

            sub_out_dir = os.path.join(output_dir, f"gender_{label}")
            metrics = lightgbm_groupcv_with_exports(
                df=sub_df,
                target_col=target_col,
                group_col=group_col,
                output_dir=sub_out_dir,
                columns=columns,
                handle_nans=handle_nans,
                impute_strategy=impute_strategy,
                n_splits=n_splits,
                random_state=random_state,
                lgbm_params=lgbm_params,
                num_boost_round=num_boost_round,
                early_stopping_rounds=early_stopping_rounds,
                shap_sample=shap_sample,
                save_plots=save_plots,
                split_gender=False,
                split_gender_post_train=False,
                regress_out_confounds=regress_out_confounds,
                confound_columns=confound_columns,
                confound_alpha=confound_alpha,
                optimize_hyperparams=optimize_hyperparams,
                n_iter_search=n_iter_search,
                validation_fraction=validation_fraction
            )

            row = {"gender": label}
            row.update(metrics)
            metrics_rows.append(row)
            results_by_gender[label] = metrics

        if metrics_rows:
            metrics_df = pd.DataFrame(metrics_rows)
            metrics_df.to_csv(
                os.path.join(output_dir, "metrics_by_gender.csv"),
                index=False
            )
            
            # Save as JSON as well
            with open(os.path.join(output_dir, "metrics_by_gender.json"), "w") as f:
                json.dump(results_by_gender, f, indent=2)
            
            print(f"Saved gender specific metrics in {output_dir}")
            print(f"  - metrics_by_gender.csv: Tabular format")
            print(f"  - metrics_by_gender.json: Detailed JSON format")
            print(f"  - gender_male/: Male-specific outputs")
            print(f"  - gender_female/: Female-specific outputs")
        else:
            print("split_gender=True but no gender subsets were created")

        return results_by_gender

    # Default LightGBM parameters
    if lgbm_params is None and not optimize_hyperparams:
        lgbm_params = {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'verbosity': -1,
            'seed': random_state,
        }
    elif lgbm_params is not None and not optimize_hyperparams:
        lgbm_params = lgbm_params.copy()
        lgbm_params['seed'] = random_state

    X_all = df.drop(columns=[target_col])
    if columns is None:
        feature_cols = X_all.select_dtypes(include=[np.number]).columns.tolist()
    else:
        missing = [c for c in columns if c not in X_all.columns]
        if missing:
            raise ValueError(f"Requested columns not in df: {missing}")
        feature_cols = columns
    if not feature_cols:
        raise ValueError("No numeric feature columns found or selected")

    # Sanitize feature column names for LightGBM (replace special JSON characters with _)
    import re
    original_feature_cols = feature_cols.copy()
    sanitized_feature_cols = [re.sub(r'[^a-zA-Z0-9_]', '_', col) for col in feature_cols]
    
    # Create mapping from original to sanitized names
    feature_name_mapping = dict(zip(original_feature_cols, sanitized_feature_cols))
    
    # Check for duplicate sanitized names
    if len(set(sanitized_feature_cols)) != len(sanitized_feature_cols):
        print("Warning: Sanitizing feature names resulted in duplicates. Adding suffixes...")
        seen = {}
        final_sanitized = []
        for orig, san in zip(original_feature_cols, sanitized_feature_cols):
            if san in seen:
                seen[san] += 1
                final_sanitized.append(f"{san}_{seen[san]}")
            else:
                seen[san] = 0
                final_sanitized.append(san)
        sanitized_feature_cols = final_sanitized
        feature_name_mapping = dict(zip(original_feature_cols, sanitized_feature_cols))
    
    feature_cols = sanitized_feature_cols

    # Build aligned dataframe
    used_cols = original_feature_cols + [target_col, group_col]
    
    # Check if research_stage exists and add it to used_cols
    has_research_stage = "research_stage" in df.columns
    if has_research_stage and "research_stage" not in used_cols:
        used_cols.append("research_stage")
    
    if regress_out_confounds:
        for conf_col in confound_columns:
            if conf_col not in used_cols:
                used_cols.append(conf_col)
    
    if split_gender_post_train:
        if "gender" not in used_cols:
            used_cols.append("gender")
    
    df_used = df[used_cols].copy()
    df_used = df_used.dropna(subset=[target_col, group_col])

    # Rename feature columns to sanitized versions
    X = df_used[original_feature_cols].copy()
    X.columns = sanitized_feature_cols
    
    y = df_used[target_col]
    groups = df_used[group_col]
    
    # Store research_stage if available
    if has_research_stage:
        research_stage_series = df_used["research_stage"]
    else:
        research_stage_series = None
    
    if split_gender_post_train:
        gender_series = df_used["gender"]
    else:
        gender_series = None
    
    if regress_out_confounds:
        confounds = df_used[confound_columns]
    else:
        confounds = None

    if handle_nans not in {"impute", "drop"}:
        raise ValueError("handle_nans must be 'impute' or 'drop'")

    if handle_nans == "drop":
        mask = ~X.isna().any(axis=1)
        if regress_out_confounds:
            mask = mask & ~confounds.isna().any(axis=1)
        if split_gender_post_train:
            mask = mask & ~gender_series.isna()
        if has_research_stage:
            mask = mask & ~research_stage_series.isna()
        X = X.loc[mask]
        y = y.loc[mask]
        groups = groups.loc[mask]
        if regress_out_confounds:
            confounds = confounds.loc[mask]
        if split_gender_post_train:
            gender_series = gender_series.loc[mask]
        if has_research_stage:
            research_stage_series = research_stage_series.loc[mask]
    elif handle_nans == "impute":
        imputer = SimpleImputer(strategy=impute_strategy)
        X_imputed = imputer.fit_transform(X)
        X = pd.DataFrame(X_imputed, index=X.index, columns=feature_cols)
        
        if regress_out_confounds:
            confounds_imputed = imputer.fit_transform(confounds)
            confounds = pd.DataFrame(confounds_imputed, index=confounds.index, columns=confound_columns)
    
    if regress_out_confounds:
        print(f"Regressing out confounds: {confound_columns}")
        
        from sklearn.preprocessing import StandardScaler
        
        # Standardize confounds and features
        confound_scaler = StandardScaler()
        confounds_scaled = confound_scaler.fit_transform(confounds)
        
        feature_scaler = StandardScaler()
        X_scaled = feature_scaler.fit_transform(X)
        
        # Regress out confounds from each feature
        ridge_confound = Ridge(alpha=confound_alpha, random_state=random_state)
        X_residuals = np.zeros_like(X_scaled)
        
        for i in range(X_scaled.shape[1]):
            y_feature = X_scaled[:, i]
            ridge_confound.fit(confounds_scaled, y_feature)
            y_pred = ridge_confound.predict(confounds_scaled)
            X_residuals[:, i] = y_feature - y_pred
        
        X = pd.DataFrame(X_residuals, index=X.index, columns=feature_cols)
        
        confound_info = {
            "confound_columns": confound_columns,
            "confound_alpha": confound_alpha,
            "n_confounds": len(confound_columns)
        }
        with open(os.path.join(output_dir, "confound_regression_info.json"), "w") as f:
            json.dump(confound_info, f, indent=2)
        
        print(f"Confounds regressed out. Features are now residuals.")

    gkf = GroupKFold(n_splits=n_splits)

    oof = pd.Series(index=y.index, dtype=float)
    fold_r2 = []
    feature_importance_list = []
    fold_best_params = []
    
    if split_gender_post_train:
        fold_metrics_by_gender = {"male": [], "female": []}

    for fold_idx, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        
        # Hyperparameter optimization
        if optimize_hyperparams:
            groups_tr = groups.iloc[tr_idx]
            unique_groups = groups_tr.unique()
            n_val_groups = max(1, int(len(unique_groups) * validation_fraction))
            
            np.random.seed(random_state + fold_idx)
            val_groups = np.random.choice(unique_groups, size=n_val_groups, replace=False)
            
            # Split training into train_inner and val_inner
            mask_val = groups_tr.isin(val_groups)
            X_tr_inner = X_tr.loc[~mask_val]
            y_tr_inner = y_tr.loc[~mask_val]
            X_val_inner = X_tr.loc[mask_val]
            y_val_inner = y_tr.loc[mask_val]
            
            # Define parameter search space
            param_distributions = {
                'num_leaves': randint(20, 150),
                'max_depth': randint(3, 15),
                'learning_rate': uniform(0.01, 0.2),
                'n_estimators': randint(100, 1000),
                'min_child_samples': randint(10, 100),
                'subsample': uniform(0.6, 0.4),  # 0.6 to 1.0
                'colsample_bytree': uniform(0.6, 0.4),  # 0.6 to 1.0
                'reg_alpha': uniform(0, 1.0),
                'reg_lambda': uniform(0, 1.0),
            }
            
            # Create LightGBM regressor
            lgbm_estimator = lgb.LGBMRegressor(
                objective='regression',
                metric='rmse',
                boosting_type='gbdt',
                verbosity=-1,
                random_state=random_state,

            )
            
            # Perform randomized search
            print(f"Fold {fold_idx}: Optimizing hyperparameters with {n_iter_search} iterations...")
            random_search = RandomizedSearchCV(
                lgbm_estimator,
                param_distributions=param_distributions,
                n_iter=n_iter_search,
                scoring='r2',
                cv=3,
                random_state=random_state + fold_idx,
                verbose=0
            )
            
            random_search.fit(X_tr_inner, y_tr_inner)
            best_params = random_search.best_params_
            best_val_score = random_search.best_score_
            
            print(f"Fold {fold_idx}: Best validation R2 = {best_val_score:.4f}")
            print(f"Fold {fold_idx}: Best params = {best_params}")
            
            fold_best_params.append(best_params)
            current_params = best_params.copy()
            current_params['objective'] = 'regression'
            current_params['metric'] = 'rmse'
            current_params['boosting_type'] = 'gbdt'
            current_params['verbosity'] = -1
            current_params['seed'] = random_state
        else:
            current_params = lgbm_params.copy()
        
        # Train model on full training fold with optimized or default params
        train_data = lgb.Dataset(X_tr, label=y_tr)
        
        if early_stopping_rounds:
            valid_data = lgb.Dataset(X_va, label=y_va, reference=train_data)
            model = lgb.train(
                current_params,
                train_data,
                num_boost_round=num_boost_round,
                valid_sets=[valid_data],
                callbacks=[lgb.early_stopping(stopping_rounds=early_stopping_rounds, verbose=False)]
            )
        else:
            model = lgb.train(
                current_params,
                train_data,
                num_boost_round=num_boost_round
            )
        
        y_pred = model.predict(X_va, num_iteration=model.best_iteration if early_stopping_rounds else None)
        oof.iloc[va_idx] = y_pred
        fold_r2.append(r2_score(y_va, y_pred))
        
        # Store feature importance
        importance = model.feature_importance(importance_type='gain')
        feature_importance_list.append(pd.DataFrame({
            'feature': feature_cols,
            'importance': importance,
            'fold': fold_idx
        }))
        
        if split_gender_post_train:
            gender_va = gender_series.iloc[va_idx]
            for gender_value, label in [(1, "male"), (0, "female")]:
                gender_mask = gender_va == gender_value
                if gender_mask.sum() > 0:
                    y_va_gender = y_va[gender_mask]
                    y_pred_gender = y_pred[gender_mask]
                    
                    gender_r2 = r2_score(y_va_gender, y_pred_gender)
                    gender_r = pearsonr(y_va_gender, y_pred_gender)[0] if np.std(y_pred_gender) > 0 else np.nan
                    gender_mae = mean_absolute_error(y_va_gender, y_pred_gender)
                    gender_rmse = mean_squared_error(y_va_gender, y_pred_gender, squared=False)
                    
                    fold_metrics_by_gender[label].append({
                        "fold": fold_idx,
                        "n_samples": int(gender_mask.sum()),
                        "R2": float(gender_r2),
                        "Pearson_r": float(gender_r),
                        "MAE": float(gender_mae),
                        "RMSE": float(gender_rmse)
                    })

    oof_r2 = r2_score(y, oof)
    oof_r = pearsonr(y, oof)[0] if np.std(oof) > 0 else np.nan
    oof_mae = mean_absolute_error(y, oof)
    oof_rmse = mean_squared_error(y, oof, squared=False)

    # Determine final parameters (average of optimized params if optimization was done)
    if optimize_hyperparams and fold_best_params:
        # Average numeric parameters
        final_params = {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'verbosity': -1,
            'seed': random_state,
        }
        
        # Average each hyperparameter across folds
        for param_key in fold_best_params[0].keys():
            values = [params[param_key] for params in fold_best_params]
            if isinstance(values[0], (int, np.integer)):
                final_params[param_key] = int(np.mean(values))
            else:
                final_params[param_key] = float(np.mean(values))
        
        print(f"Final model params (averaged): {final_params}")
    else:
        final_params = lgbm_params

    # Train final model on all data
    train_data_full = lgb.Dataset(X, label=y)
    final_model = lgb.train(
        final_params,
        train_data_full,
        num_boost_round=num_boost_round
    )
    
    # Save final model
    final_model.save_model(os.path.join(output_dir, "model.txt"))
    
    # Save feature name mapping (sanitized -> original)
    reverse_mapping = {v: k for k, v in feature_name_mapping.items()}
    mapping_df = pd.DataFrame({
        'original_name': original_feature_cols,
        'sanitized_name': sanitized_feature_cols
    })
    mapping_df.to_csv(os.path.join(output_dir, "feature_name_mapping.csv"), index=False)

    # Aggregate feature importance
    feature_importance_df = pd.concat(feature_importance_list, axis=0)
    feature_importance_summary = feature_importance_df.groupby('feature')['importance'].agg(['mean', 'std']).reset_index()
    
    # Add original feature names back
    feature_importance_summary['original_feature'] = feature_importance_summary['feature'].map(reverse_mapping)
    feature_importance_summary = feature_importance_summary[['original_feature', 'feature', 'mean', 'std']]
    feature_importance_summary = feature_importance_summary.sort_values('mean', ascending=False)
    feature_importance_summary.to_csv(os.path.join(output_dir, "feature_importance.csv"), index=False)

    # Save predictions
    pred_df = pd.DataFrame({
        "index": oof.index.astype(str),
        "group": groups.astype(str),
        "true_values": y.values,
        "predictions": oof.values,
    })
    fold_marks = pd.Series(index=y.index, dtype="Int64")
    for fold_idx, (_, va_idx) in enumerate(gkf.split(X, y, groups), start=1):
        fold_marks.iloc[va_idx] = fold_idx
    pred_df["fold"] = fold_marks.values
    
    # Add research_stage if available
    if has_research_stage:
        pred_df["research_stage"] = research_stage_series.values
    
    if split_gender_post_train:
        pred_df["gender"] = gender_series.values
    
    pred_df.to_csv(os.path.join(output_dir, "predictions.csv"), index=False)

    # Gender-specific metrics if split_gender_post_train
    if split_gender_post_train:
        gender_metrics_overall = {}
        
        for gender_value, label in [(1, "male"), (0, "female")]:
            gender_mask = gender_series == gender_value
            if gender_mask.sum() > 0:
                y_gender = y[gender_mask]
                oof_gender = oof[gender_mask]
                
                gender_r2 = r2_score(y_gender, oof_gender)
                gender_r = pearsonr(y_gender, oof_gender)[0] if np.std(oof_gender) > 0 else np.nan
                gender_mae = mean_absolute_error(y_gender, oof_gender)
                gender_rmse = mean_squared_error(y_gender, oof_gender, squared=False)
                
                gender_metrics_overall[label] = {
                    "n_samples": int(gender_mask.sum()),
                    "oof_Pearson_r": float(gender_r),
                    "oof_R2": float(gender_r2),
                    "oof_MAE": float(gender_mae),
                    "oof_RMSE": float(gender_rmse),
                    "per_fold_metrics": fold_metrics_by_gender[label]
                }
        
        with open(os.path.join(output_dir, "metrics_by_gender.json"), "w") as f:
            json.dump(gender_metrics_overall, f, indent=2)
        
        gender_summary = []
        for label, metrics in gender_metrics_overall.items():
            row = {"gender": label}
            row.update({k: v for k, v in metrics.items() if k != "per_fold_metrics"})
            gender_summary.append(row)
        
        if gender_summary:
            pd.DataFrame(gender_summary).to_csv(
                os.path.join(output_dir, "metrics_by_gender_summary.csv"),
                index=False
            )
        
        print(f"Saved gender-specific evaluation metrics")

    metrics = {
        "oof_Pearson_r": float(oof_r),
        "oof_R2": float(oof_r2),
        "oof_MAE": float(oof_mae),
        "oof_RMSE": float(oof_rmse),
        "per_fold_R2": [float(x) for x in fold_r2],
        "n_samples": int(X.shape[0]),
        "n_features": int(X.shape[1]),
        "lgbm_params": final_params,
        "num_boost_round": num_boost_round,
        "handle_nans": handle_nans,
        "impute_strategy": impute_strategy,
        "split_gender_post_train": bool(split_gender_post_train),
        "regress_out_confounds": bool(regress_out_confounds),
        "confound_columns": confound_columns if regress_out_confounds else None,
        "confound_alpha": float(confound_alpha) if regress_out_confounds else None,
        "optimize_hyperparams": bool(optimize_hyperparams),
        "fold_best_params": fold_best_params if optimize_hyperparams else None
    }
    with open(os.path.join(output_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    # SHAP analysis
    try:
        import shap
        if shap_sample and X.shape[0] > shap_sample:
            bg_idx = np.random.RandomState(123).choice(
                X.index, shap_sample, replace=False
            )
            X_bg = X.loc[bg_idx]
        else:
            X_bg = X

        explainer = shap.TreeExplainer(final_model)
        shap_values = explainer(X_bg)

        if save_plots:
            # Create a copy with original feature names for better readability
            X_bg_display = X_bg.copy()
            X_bg_display.columns = [reverse_mapping.get(col, col) for col in X_bg.columns]
            
            plt.figure(figsize=(10, 8))
            shap.summary_plot(
                shap_values.values,
                features=X_bg_display,
                feature_names=original_feature_cols,
                show=False
            )
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, "shap_summary.png"), dpi=200)
            plt.close()

    except Exception as e:
        with open(os.path.join(output_dir, "shap_error.txt"), "w") as f:
            f.write(str(e))

    print(f"Saved outputs in {output_dir}")
    return metrics

In [186]:
def run_multi_seed_lightgbm(
    df: pd.DataFrame,
    target_col: str,
    group_col: str,
    output_dir: str,
    seeds: list[int] = [42, 1, 2, 3, 4],
    **kwargs
):
    """
    Wrapper to run lightgbm_groupcv_with_exports multiple times with different seeds.
    It aggregates predictions (bagging) to reduce variance and calculates 
    metrics on the averaged predictions.

    Parameters
    ----------
    df, target_col, group_col, output_dir : Same as original function.
    seeds : list[int]
        List of random seeds to run.
    **kwargs : 
        All other arguments to pass to lightgbm_groupcv_with_exports 
        (e.g. columns, lgbm_params, split_gender_post_train, split_gender, etc.)
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Check if split_gender is enabled
    split_gender = kwargs.get('split_gender', False)
    
    print(f"Starting Multi-Seed LightGBM Run with seeds: {seeds}")
    print("="*60)

    # Handle split_gender separately
    if split_gender:
        print("split_gender=True: Processing each gender separately across seeds")
        
        # Process each gender separately
        gender_results = {}
        for gender_value, gender_label in [(1, "male"), (0, "female")]:
            print(f"\n{'='*60}")
            print(f"Processing gender: {gender_label}")
            print(f"{'='*60}")
            
            all_pred_dfs = []
            all_importance_dfs = []
            seed_metrics_list = []
            
            # Loop through seeds for this gender
            for seed in seeds:
                print(f"--> Running Seed: {seed} for {gender_label}")
                
                # Create a subdirectory for this seed
                seed_dir = os.path.join(output_dir, f"seed_{seed}")
                
                # Update random_state in kwargs
                run_kwargs = kwargs.copy()
                run_kwargs['random_state'] = seed
                
                # Run the original function
                _ = lightgbm_groupcv_with_exports(
                    df=df,
                    target_col=target_col,
                    group_col=group_col,
                    output_dir=seed_dir,
                    **run_kwargs
                )
                
                # Read back the predictions for this seed and gender
                gender_pred_path = os.path.join(seed_dir, f"gender_{gender_label}", "predictions.csv")
                if os.path.exists(gender_pred_path):
                    seed_pred_df = pd.read_csv(gender_pred_path)
                    
                    # Ensure index is treated as string for consistent merging
                    seed_pred_df['index'] = seed_pred_df['index'].astype(str)
                    seed_pred_df = seed_pred_df.set_index('index')
                    
                    # Keep only relevant columns from the first seed
                    if len(all_pred_dfs) == 0:
                        base_columns = [c for c in seed_pred_df.columns if c != 'predictions' and c != 'fold']
                        base_df = seed_pred_df[base_columns].copy()
                    
                    # Extract just the prediction for this seed
                    pred_series = seed_pred_df['predictions'].rename(f"pred_seed_{seed}")
                    all_pred_dfs.append(pred_series)
                    
                    # Calculate metrics for this seed
                    y_true_seed = seed_pred_df['true_values']
                    y_pred_seed = seed_pred_df['predictions']
                    mask_seed = ~y_true_seed.isna() & ~y_pred_seed.isna()
                    y_true_seed = y_true_seed[mask_seed]
                    y_pred_seed = y_pred_seed[mask_seed]
                    
                    seed_r2 = r2_score(y_true_seed, y_pred_seed)
                    seed_r = pearsonr(y_true_seed, y_pred_seed)[0] if np.std(y_pred_seed) > 0 else np.nan
                    seed_mae = mean_absolute_error(y_true_seed, y_pred_seed)
                    seed_rmse = mean_squared_error(y_true_seed, y_pred_seed, squared=False)
                    
                    seed_metrics_list.append({
                        'seed': seed,
                        'R2': float(seed_r2),
                        'Pearson_r': float(seed_r),
                        'MAE': float(seed_mae),
                        'RMSE': float(seed_rmse)
                    })
                
                # Read back the feature importance for this seed and gender
                gender_importance_path = os.path.join(seed_dir, f"gender_{gender_label}", "feature_importance.csv")
                if os.path.exists(gender_importance_path):
                    seed_importance_df = pd.read_csv(gender_importance_path)
                    seed_importance_df['seed'] = seed
                    all_importance_dfs.append(seed_importance_df)
            
            if not all_pred_dfs:
                print(f"Warning: No predictions found for {gender_label}")
                continue
            
            print(f"Aggregating results for {gender_label}...")
            
            # Aggregate Predictions (Averaging)
            preds_concat = pd.concat(all_pred_dfs, axis=1)
            
            # Calculate the mean prediction across rows (seeds)
            base_df['mean_predictions'] = preds_concat.mean(axis=1)
            
            # Determine std dev of predictions across seeds (uncertainty measure)
            base_df['pred_std'] = preds_concat.std(axis=1)
            
            # Merge the individual seed predictions back in for reference
            final_pred_df = pd.concat([base_df, preds_concat], axis=1)
            
            # Save averaged predictions for this gender
            gender_out_dir = os.path.join(output_dir, f"gender_{gender_label}")
            os.makedirs(gender_out_dir, exist_ok=True)
            final_pred_path = os.path.join(gender_out_dir, "predictions_averaged.csv")
            final_pred_df.reset_index().to_csv(final_pred_path, index=False)
            
            # Recalculate Metrics on the Averaged Prediction
            y_true = final_pred_df['true_values']
            y_pred = final_pred_df['mean_predictions']
            
            # Drop NaNs if any
            mask = ~y_true.isna() & ~y_pred.isna()
            y_true = y_true[mask]
            y_pred = y_pred[mask]

            r2 = r2_score(y_true, y_pred)
            pearson_r = pearsonr(y_true, y_pred)[0] if np.std(y_pred) > 0 else np.nan
            mae = mean_absolute_error(y_true, y_pred)
            rmse = mean_squared_error(y_true, y_pred, squared=False)
            
            # Calculate statistics across seeds
            seed_metrics_df = pd.DataFrame(seed_metrics_list)
            
            gender_metrics = {
                "gender": gender_label,
                "n_seeds": len(seeds),
                "seeds_used": seeds,
                "averaged_Pearson_r": float(pearson_r),
                "averaged_R2": float(r2),
                "averaged_MAE": float(mae),
                "averaged_RMSE": float(rmse),
                "n_samples": int(len(y_true)),
                # Per-seed statistics
                "per_seed_R2_mean": float(seed_metrics_df['R2'].mean()),
                "per_seed_R2_std": float(seed_metrics_df['R2'].std()),
                "per_seed_Pearson_r_mean": float(seed_metrics_df['Pearson_r'].mean()),
                "per_seed_Pearson_r_std": float(seed_metrics_df['Pearson_r'].std()),
                "per_seed_MAE_mean": float(seed_metrics_df['MAE'].mean()),
                "per_seed_MAE_std": float(seed_metrics_df['MAE'].std()),
                "per_seed_RMSE_mean": float(seed_metrics_df['RMSE'].mean()),
                "per_seed_RMSE_std": float(seed_metrics_df['RMSE'].std()),
                "per_seed_metrics": seed_metrics_list
            }
            
            gender_results[gender_label] = gender_metrics
            
            # Save gender-specific metrics JSON
            with open(os.path.join(gender_out_dir, "metrics_averaged.json"), "w") as f:
                json.dump(gender_metrics, f, indent=2)
            
            # Save per-seed metrics as CSV
            seed_metrics_df.to_csv(os.path.join(gender_out_dir, "metrics_per_seed.csv"), index=False)
            
            # Aggregate Feature Importance for this gender
            if all_importance_dfs:
                all_importance = pd.concat(all_importance_dfs, axis=0)
                importance_summary = all_importance.groupby("original_feature")["mean"].agg(['mean', 'std', 'count']).reset_index()
                importance_summary.columns = ['feature', 'importance_mean', 'importance_std', 'count']
                importance_summary['abs_mean_importance'] = importance_summary['importance_mean'].abs()
                importance_summary = importance_summary.sort_values("abs_mean_importance", ascending=False)
                
                importance_summary.to_csv(os.path.join(gender_out_dir, "feature_importance_averaged.csv"), index=False)
            
            print(f"Done with {gender_label}. Averaged R2: {r2:.4f} (Per-seed mean: {gender_metrics['per_seed_R2_mean']:.4f} ± {gender_metrics['per_seed_R2_std']:.4f})")
        
        # Save combined gender metrics
        if gender_results:
            gender_df = pd.DataFrame(gender_results).T
            gender_df.to_csv(os.path.join(output_dir, "metrics_averaged_by_gender.csv"))
            
            with open(os.path.join(output_dir, "metrics_averaged_by_gender.json"), "w") as f:
                json.dump(gender_results, f, indent=2)
            
            print(f"\n{'='*60}")
            print(f"All gender-specific results saved in {output_dir}")
            print(f"  - gender_male/: male-specific averaged outputs")
            print(f"  - gender_female/: female-specific averaged outputs")
            print(f"  - metrics_averaged_by_gender.csv: combined summary")
            print(f"  - metrics_averaged_by_gender.json: structured format")
        
        return gender_results
    
    # Original logic for non-split_gender case
    all_pred_dfs = []
    all_importance_dfs = []
    seed_metrics_list = []

    # 1. Loop through seeds and run the base pipeline
    for seed in seeds:
        print(f"--> Running Seed: {seed}")
        
        # Create a subdirectory for this seed to avoid file collisions
        seed_dir = os.path.join(output_dir, f"seed_{seed}")
        
        # Update random_state in kwargs
        run_kwargs = kwargs.copy()
        run_kwargs['random_state'] = seed
        
        # Run the original function
        _ = lightgbm_groupcv_with_exports(
            df=df,
            target_col=target_col,
            group_col=group_col,
            output_dir=seed_dir,
            **run_kwargs
        )
        
        # Read back the predictions for this seed
        pred_path = os.path.join(seed_dir, "predictions.csv")
        if os.path.exists(pred_path):
            # Read, set index to align rows, rename prediction column
            seed_pred_df = pd.read_csv(pred_path)
            
            # Ensure index is treated as string for consistent merging
            seed_pred_df['index'] = seed_pred_df['index'].astype(str)
            seed_pred_df = seed_pred_df.set_index('index')
            
            # Keep only relevant columns from the first seed
            if len(all_pred_dfs) == 0:
                base_columns = [c for c in seed_pred_df.columns if c != 'predictions' and c != 'fold']
                base_df = seed_pred_df[base_columns].copy()
            
            # Extract just the prediction for this seed
            pred_series = seed_pred_df['predictions'].rename(f"pred_seed_{seed}")
            all_pred_dfs.append(pred_series)
            
            # Calculate metrics for this seed
            y_true_seed = seed_pred_df['true_values']
            y_pred_seed = seed_pred_df['predictions']
            mask_seed = ~y_true_seed.isna() & ~y_pred_seed.isna()
            y_true_seed = y_true_seed[mask_seed]
            y_pred_seed = y_pred_seed[mask_seed]
            
            seed_r2 = r2_score(y_true_seed, y_pred_seed)
            seed_r = pearsonr(y_true_seed, y_pred_seed)[0] if np.std(y_pred_seed) > 0 else np.nan
            seed_mae = mean_absolute_error(y_true_seed, y_pred_seed)
            seed_rmse = mean_squared_error(y_true_seed, y_pred_seed, squared=False)
            
            seed_metrics_list.append({
                'seed': seed,
                'R2': float(seed_r2),
                'Pearson_r': float(seed_r),
                'MAE': float(seed_mae),
                'RMSE': float(seed_rmse)
            })
        
        # Read back the feature importance for this seed
        importance_path = os.path.join(seed_dir, "feature_importance.csv")
        if os.path.exists(importance_path):
            seed_importance_df = pd.read_csv(importance_path)
            seed_importance_df['seed'] = seed
            all_importance_dfs.append(seed_importance_df)

    print("="*60)
    print("Aggregating results...")

    # 2. Aggregate Predictions (Averaging)
    # Concatenate all prediction series along columns, aligning by index
    preds_concat = pd.concat(all_pred_dfs, axis=1)
    
    # Calculate the mean prediction across rows (seeds)
    base_df['mean_predictions'] = preds_concat.mean(axis=1)
    
    # Determine std dev of predictions across seeds (uncertainty measure)
    base_df['pred_std'] = preds_concat.std(axis=1)
    
    # Merge the individual seed predictions back in for reference
    final_pred_df = pd.concat([base_df, preds_concat], axis=1)
    
    # Save averaged predictions
    final_pred_path = os.path.join(output_dir, "predictions_averaged.csv")
    final_pred_df.reset_index().to_csv(final_pred_path, index=False)
    
    # 3. Recalculate Metrics on the Averaged Prediction
    y_true = final_pred_df['true_values']
    y_pred = final_pred_df['mean_predictions']
    
    # Drop NaNs if any
    mask = ~y_true.isna() & ~y_pred.isna()
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    r2 = r2_score(y_true, y_pred)
    pearson_r = pearsonr(y_true, y_pred)[0] if np.std(y_pred) > 0 else np.nan
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    
    # Calculate statistics across seeds
    seed_metrics_df = pd.DataFrame(seed_metrics_list)
    
    metrics = {
        "n_seeds": len(seeds),
        "seeds_used": seeds,
        "averaged_Pearson_r": float(pearson_r),
        "averaged_R2": float(r2),
        "averaged_MAE": float(mae),
        "averaged_RMSE": float(rmse),
        # Per-seed statistics
        "per_seed_R2_mean": float(seed_metrics_df['R2'].mean()),
        "per_seed_R2_std": float(seed_metrics_df['R2'].std()),
        "per_seed_Pearson_r_mean": float(seed_metrics_df['Pearson_r'].mean()),
        "per_seed_Pearson_r_std": float(seed_metrics_df['Pearson_r'].std()),
        "per_seed_MAE_mean": float(seed_metrics_df['MAE'].mean()),
        "per_seed_MAE_std": float(seed_metrics_df['MAE'].std()),
        "per_seed_RMSE_mean": float(seed_metrics_df['RMSE'].mean()),
        "per_seed_RMSE_std": float(seed_metrics_df['RMSE'].std()),
        "per_seed_metrics": seed_metrics_list
    }

    # 4. Handle Gender Splitting Metrics (if applicable)
    split_gender_post = kwargs.get('split_gender_post_train', False)
    if split_gender_post and 'gender' in final_pred_df.columns:
        gender_metrics = {}
        for gender_val in final_pred_df['gender'].unique():
            label = str(gender_val)
            if label == "1" or label == "1.0": label = "male"
            if label == "0" or label == "0.0": label = "female"
            
            sub_df = final_pred_df[final_pred_df['gender'] == gender_val]
            if len(sub_df) > 0:
                y_g = sub_df['true_values']
                p_g = sub_df['mean_predictions']
                
                g_r2 = r2_score(y_g, p_g)
                g_r = pearsonr(y_g, p_g)[0] if np.std(p_g) > 0 else np.nan
                g_mae = mean_absolute_error(y_g, p_g)
                g_rmse = mean_squared_error(y_g, p_g, squared=False)
                
                gender_metrics[label] = {
                    "R2": float(g_r2),
                    "Pearson_r": float(g_r),
                    "MAE": float(g_mae),
                    "RMSE": float(g_rmse),
                    "n_samples": int(len(sub_df))
                }
        
        metrics['gender_metrics'] = gender_metrics
        
        # Save gender summary CSV
        pd.DataFrame(gender_metrics).T.to_csv(
            os.path.join(output_dir, "metrics_averaged_by_gender.csv")
        )

    # Save final metrics JSON
    with open(os.path.join(output_dir, "metrics_averaged.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    
    # Save per-seed metrics as CSV
    seed_metrics_df.to_csv(os.path.join(output_dir, "metrics_per_seed.csv"), index=False)

    # 5. Aggregate Feature Importance
    if all_importance_dfs:
        all_importance = pd.concat(all_importance_dfs, axis=0)
        # Group by feature name and calculate statistics
        importance_summary = all_importance.groupby("original_feature")["mean"].agg(['mean', 'std', 'count']).reset_index()
        importance_summary.columns = ['feature', 'importance_mean', 'importance_std', 'count']
        importance_summary['abs_mean_importance'] = importance_summary['importance_mean'].abs()
        importance_summary = importance_summary.sort_values("abs_mean_importance", ascending=False)
        
        importance_summary.to_csv(os.path.join(output_dir, "feature_importance_averaged.csv"), index=False)

    print(f"Done. Averaged results saved to {output_dir}")
    print(f"Final Averaged R2: {r2:.4f}")
    print(f"Per-seed R2: {metrics['per_seed_R2_mean']:.4f} ± {metrics['per_seed_R2_std']:.4f}")
    
    return metrics

In [55]:
lightgbm_groupcv_with_exports(
    df=metabolomics_full,
    target_col='age',
    group_col='subject_number',
    output_dir=DeepFolderPath + 'LGBM_stuff/lightgbm_Metabo_age_prediction',
    columns=metabolomics_data.columns.to_list(),
    handle_nans='impute',
    impute_strategy='median',
    n_splits=5, 
    random_state=42,
    optimize_hyperparams=False,
    n_iter_search=10,
    split_gender=True
    )

Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length//lightgbm_Metabo_age_prediction/gender_male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length//lightgbm_Metabo_age_prediction/gender_female
Saved gender specific metrics in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length//lightgbm_Metabo_age_prediction
  - metrics_by_gender.csv: Tabular format
  - metrics_by_gender.json: Detailed JSON format
  - gender_male/: Male-specific outputs
  - gender_female/: Female-specific outputs
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length//lightgbm_Metabo_age_prediction/gender_female
Saved gender specific metrics in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length//lightgbm_Metabo_age_prediction
  - metrics_by_gender.csv: Tabular format
  - metrics_by_gender.json: Detailed JSON format
  

{'male': {'oof_Pearson_r': 0.6756239164973167,
  'oof_R2': 0.4519981600076708,
  'oof_MAE': 4.552606663476216,
  'oof_RMSE': 5.674242394515507,
  'per_fold_R2': [0.505361699342181,
   0.44186099062894046,
   0.4431861147994709,
   0.42805481063304185,
   0.4388744951985465],
  'n_samples': 5342,
  'n_features': 2064,
  'lgbm_params': {'objective': 'regression',
   'metric': 'rmse',
   'boosting_type': 'gbdt',
   'verbosity': -1,
   'seed': 42},
  'num_boost_round': 100,
  'handle_nans': 'impute',
  'impute_strategy': 'median',
  'split_gender_post_train': False,
  'regress_out_confounds': False,
  'confound_columns': None,
  'confound_alpha': None,
  'optimize_hyperparams': False,
  'fold_best_params': None},
 'female': {'oof_Pearson_r': 0.7776821398215805,
  'oof_R2': 0.6037700357172394,
  'oof_MAE': 3.836981421913306,
  'oof_RMSE': 4.845146312313372,
  'per_fold_R2': [0.5838314315550189,
   0.6305974315075193,
   0.6115303089825056,
   0.5984753694851614,
   0.5918968627603229],
  'n

In [192]:
datasets_lgbm = {
    'sleep': (sleep_full, sleep_data.columns.tolist()),
    'metabolomics': (metabolomics_full, metabolomics_data.columns.tolist()),
    'diet': (diet_full, diet_data.columns.tolist()),
    'microbiome': (microbiome_full_filtered, microbiome_data_filtered.columns.tolist()),
    'retina': (retina_full, retina_data.columns.tolist()),
    'NMR': (NMR_full, NMR_cols),
    'DEXA': (dxa_full, dxa_data.columns.tolist()),
    'blood_test': (bt_full, bt_data.columns.tolist()),
    'diet': (diet_full, diet_data.columns.tolist()),
    # 'lifestyle': (lifestyle_full, lifestyle_data.columns.tolist())
}

In [ ]:

for dataset_name, (df, feature_cols) in datasets_lgbm.items():
    print(f"\n{'='*60}")
    print(f"Running LightGBM for {dataset_name.upper()}")
    print(f"{'='*60}")
    
    run_multi_seed_lightgbm(
        df=df,
        target_col='age',
        group_col='subject_number',
        output_dir=DeepFolderPath + f'LGBM_stuff_new/multi_seed_lgbm_age_prediction_{dataset_name}',
        columns=feature_cols,
        handle_nans='impute',
        impute_strategy='median',
        n_splits=5,
        seeds=[42, 1, 2, 3, 4, 17, 99, 123, 256, 512],
        save_plots=True,
        split_gender=True
    )
    
    print(f"Completed {dataset_name} - Results saved")


Running LightGBM for SLEEP
Starting Multi-Seed LightGBM Run with seeds: [42, 1, 2, 3, 4, 17, 99, 123, 256, 512]
split_gender=True: Processing each gender separately across seeds

Processing gender: male
--> Running Seed: 42 for male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/LGBM_stuff_new/multi_seed_lgbm_age_prediction_sleep/seed_42/gender_male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/LGBM_stuff_new/multi_seed_lgbm_age_prediction_sleep/seed_42/gender_male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/LGBM_stuff_new/multi_seed_lgbm_age_prediction_sleep/seed_42/gender_female
Saved gender specific metrics in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/LGBM_stuff_new/multi_seed_lgbm_age_prediction_sleep/seed_42
  - metrics_by_gender.csv: Tabular format
  - metrics_by_gender.json: De

In [193]:
for dataset_name, (df, feature_cols) in datasets_lgbm.items():
    print(f"\n{'='*60}")
    print(f"Running LightGBM for {dataset_name.upper()}")
    print(f"{'='*60}")
    
    run_multi_seed_ridge(
        df=df,
        target_col='age',
        group_col='subject_number',
        output_dir=DeepFolderPath + f'Ridge_stuff_new/multi_seed_ridge_age_prediction_{dataset_name}',
        columns=feature_cols,
        handle_nans='impute',
        impute_strategy='median',
        n_splits=5,
        seeds=[42, 1, 2, 3, 4, 17, 99, 123, 256, 512],
        save_plots=True,
        split_gender=True
    )
    
    print(f"Completed {dataset_name} - Results saved")


Running LightGBM for SLEEP
Starting Multi-Seed Run with seeds: [42, 1, 2, 3, 4, 17, 99, 123, 256, 512]
split_gender=True: Processing each gender separately across seeds

Processing gender: male
--> Running Seed: 42 for male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff_new/multi_seed_ridge_age_prediction_sleep/seed_42/gender_male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff_new/multi_seed_ridge_age_prediction_sleep/seed_42/gender_male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff_new/multi_seed_ridge_age_prediction_sleep/seed_42/gender_female
Saved gender specific metrics in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff_new/multi_seed_ridge_age_prediction_sleep/seed_42
  - metrics_by_gender.csv: tabular format
  - metrics_by_gender.json: str

In [195]:
NMR_full.columns.to_list()

['subject_number',
 'research_stage',
 'Acetate',
 'Acetoacetate',
 'Acetone',
 'Ala',
 'Albumin',
 'ApoA1',
 'ApoB',
 'ApoB_by_ApoA1',
 'Cholines',
 'Citrate',
 'Clinical_LDL_C',
 'DHA',
 'DHA_pct',
 'Glucose',
 'GlycA',
 'HDL_C',
 'HDL_CE',
 'HDL_FC',
 'HDL_L',
 'HDL_P',
 'HDL_PL',
 'HDL_TG',
 'HDL_size',
 'His',
 'IDL_C',
 'IDL_CE',
 'IDL_CE_pct',
 'IDL_C_pct',
 'IDL_FC',
 'IDL_FC_pct',
 'IDL_L',
 'IDL_P',
 'IDL_PL',
 'IDL_PL_pct',
 'IDL_TG',
 'IDL_TG_pct',
 'Ile',
 'LA',
 'LA_pct',
 'LDL_C',
 'LDL_CE',
 'LDL_FC',
 'LDL_L',
 'LDL_P',
 'LDL_PL',
 'LDL_TG',
 'LDL_size',
 'L_HDL_C',
 'L_HDL_CE',
 'L_HDL_CE_pct',
 'L_HDL_C_pct',
 'L_HDL_FC',
 'L_HDL_FC_pct',
 'L_HDL_L',
 'L_HDL_P',
 'L_HDL_PL',
 'L_HDL_PL_pct',
 'L_HDL_TG',
 'L_HDL_TG_pct',
 'L_LDL_C',
 'L_LDL_CE',
 'L_LDL_CE_pct',
 'L_LDL_C_pct',
 'L_LDL_FC',
 'L_LDL_FC_pct',
 'L_LDL_L',
 'L_LDL_P',
 'L_LDL_PL',
 'L_LDL_PL_pct',
 'L_LDL_TG',
 'L_LDL_TG_pct',
 'L_VLDL_C',
 'L_VLDL_CE',
 'L_VLDL_CE_pct',
 'L_VLDL_C_pct',
 'L_VLDL_FC',
 '

In [83]:
lifestyle_data

lifestyle_accommodation_type  \
subject_number research_stage                                 
1000942861     01_00_call                               1.0   
               02_01_call                               NaN   
1001201093     baseline                                 NaN   
               01_00_call                               2.0   
               02_00_visit                              2.0   
...                                                     ...   
9999226141     02_00_visit                              1.0   
               03_00_call                               1.0   
               04_00_visit                              1.0   
9999409119     baseline                                 2.0   
               03_00_call                               2.0   

                               lifestyle_add_salt_to_food  \
subject_number research_stage                               
1000942861     01_00_call                             NaN   
               02_01_call                             NaN   
1001201093     baseline                               NaN   
               01_00_call                             NaN   
               02_00_visit                            NaN   
...                                                   ...   
9999226141     02_00_visit                            NaN   
               03_00_call                             NaN   
               04_00_visit                            1.0   
9999409119     baseline                               NaN   
               03_00_call                             NaN   

                               lifestyle_age_last_smoking_regularly1  \
subject_number research_stage                                          
1000942861     01_00_call                                        NaN   
               02_01_call                                        NaN   
1001201093     baseline                                          NaN   
               01_00_call                                        NaN   
               02_00_visit                                       NaN   
...                                                              ...   
9999226141     02_00_visit                                       NaN   
               03_00_call                                        NaN   
               04_00_visit                                       NaN   
9999409119     baseline                                          NaN   
               03_00_call                                        NaN   

                               lifestyle_age_last_smoking_regularly_age  \
subject_number research_stage                                             
1000942861     01_00_call                                           NaN   
               02_01_call                                           NaN   
1001201093     baseline                                             NaN   
               01_00_call                                           NaN   
               02_00_visit                                          NaN   
...                                                                 ...   
9999226141     02_00_visit                                          NaN   
               03_00_call                                           NaN   
               04_00_visit                                          NaN   
9999409119     baseline                                             NaN   
               03_00_call                                          26.0   

                               lifestyle_alcohol_drink  \
subject_number research_stage                            
1000942861     01_00_call                          NaN   
               02_01_call                          NaN   
1001201093     baseline                            1.0   
               01_00_call                          NaN   
               02_00_visit                         1.0   
...                                                ...   
9999226141     02_00_visit              

In [82]:
# show cols in lifestyle_data that contain smoking or tobacco related terms
smoking_cols = [col for col in lifestyle_data.columns if 'smok' in col.lower() or 'tobacco' in col.lower()]
print("Lifestyle data columns related to smoking/tobacco:")
for col in smoking_cols:
    print(f" - {col}")

Lifestyle data columns related to smoking/tobacco:
 - lifestyle_age_last_smoking_regularly1
 - lifestyle_age_last_smoking_regularly_age
 - lifestyle_easy_go_without_smoking_day
 - lifestyle_may_start_smoking_again
 - lifestyle_smoke_exposed_at_home_hours_week
 - lifestyle_smoke_exposed_outside_hours_per_week
 - lifestyle_smoke_houshold
 - lifestyle_smoke_tobacco_now
 - lifestyle_smoked_at_least_100
 - lifestyle_smoking_after_waking
 - lifestyle_smoking_cigaretts_past
 - lifestyle_smoking_compared_10years
 - lifestyle_smoking_first_age
 - lifestyle_smoking_first_age_age
 - lifestyle_smoking_stop_more_then_6months
 - lifestyle_start_smoking_age
 - lifestyle_stop_smoking_try_number
 - lifestyle_stop_smoking_try_number_number
 - lifestyle_tobacco_past_how_often
 - lifestyle_tried_give_up_smoking
 - lifestyle_want_to_stop_smoking
 - lifestyle_smoking_reduce_why__As a medical precaution
 - lifestyle_smoking_reduce_why__Doctor's recommendation
 - lifestyle_smoking_reduce_why__Economic reasons

In [102]:
from  LabData.DataLoaders import LifeStyleLoader
lifestyle_hebrew = LifeStyleLoader.LifeStyleLoader().get_data(df='hebrew')
lifestyle_english = LifeStyleLoader.LifeStyleLoader().get_data(df='english')
lifestyle_numbers = LifeStyleLoader.LifeStyleLoader().get_data(df='number')

In [ ]:
lifestyle_hebrew.df..value_counts(dropna=False)


In [130]:
lifestyle_hebrew.df.smoke_tobacco_now.value_counts(dropna=False)


smoke_tobacco_now
לא                       24265
NaN                       4140
רק לפעמים                 1868
כן, ברוב או בכל הימים     1542
מעדיפ/ה לא לענות           130
Name: count, dtype: int64

In [156]:
lifestyle_numbers.df.tobacco_past_how_often.value_counts(dropna=False)

tobacco_past_how_often
 0.0    13830
 NaN     5133
 2.0     5018
 3.0     4348
 1.0     3349
-1.0      267
Name: count, dtype: int64

In [155]:
lifestyle_hebrew.df.tobacco_past_how_often.value_counts(dropna=False)

tobacco_past_how_often
מעולם לא עישנתי              13830
NaN                           5133
עישנתי לפעמים                 5018
עישנתי ברוב או בכל הימים      4348
ניסיתי פעם או פעמיים בלבד     3349
מעדיפ/ה לא לענות               267
Name: count, dtype: int64

In [126]:
# create past_smoker boolean column in lifestyle_numbers where (2 or 3) is 1 (0 or 1 is 0), and rest is NaN
def map_past_smoker(x):
    if x in [2, 3]:
        return 1
    elif x in [0, 1]:
        return 0
    else:
        return np.nan


In [131]:
# create current_smoker boolean column in lifestyle_numbers where (1 or 0.5) is 1 (0 is 0), and rest is NaN
def map_current_smoker(x):
    if x in [1, 0.5]:
        return 1
    elif x == 0:
        return 0
    else:
        return np.nan

In [110]:
lifestyle_numbers_df = lifestyle_numbers.df.join(lifestyle_numbers.df_metadata['research_stage']).reset_index()

In [132]:
lifestyle_numbers_df['past_smoker'] = lifestyle_numbers_df['tobacco_past_how_often'].apply(map_past_smoker)
lifestyle_numbers_df['current_smoker'] = lifestyle_numbers_df['smoke_tobacco_now'].apply(map_current_smoker)

In [133]:
lifestyle_numbers_df.past_smoker.value_counts(dropna=False)

past_smoker
0.0    17179
1.0     9366
NaN     5400
Name: count, dtype: int64

In [134]:
lifestyle_numbers_df.current_smoker.value_counts(dropna=False)

current_smoker
0.0    24265
NaN     4270
1.0     3410
Name: count, dtype: int64

In [135]:
# create index column by RegistrationCode splitting at '_' and taking second part added with research_stage
lifestyle_numbers_df['index'] = lifestyle_numbers_df.apply(lambda row: f"{row['RegistrationCode'].split('_')[1]}_{row['research_stage']}", axis=1)
lifestyle_numbers_df.set_index('index', inplace=True)

In [136]:
lifestyle_numbers_df

,RegistrationCode,Date,accommodation_type,accommodation_years,add_salt_to_food,age_last_smoking_regularly1,age_last_smoking_regularly_age,alcohol_drink,alcohol_drink_past,beer_cider_pints_month,...,transportaion_to_work__Electric scooter,transportaion_to_work__Train,transportaion_to_work__Walk,transportaion_to_work__Walking,transportaion_to_work__no,transportaion_to_work__prefer not to answer,"transportaion_to_work__אף אחד מהנ""ל",research_stage,past_smoker,current_smoker
index,,,,,,,,,,,,,,,,,,,,,
1000942861_01_00_call,10K_1000942861,2023-03-20 08:01:31.157109,1.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,01_00_call,1.0,0.0
1000942861_02_01_call,10K_1000942861,2023-12-26 06:15:29.926993,1.0,22.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,02_01_call,NaN,0.0
1001201093_baseline,10K_1001201093,2021-08-25 19:13:51.000000,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,baseline,0.0,1.0
1001201093_01_00_call,10K_1001201093,2022-08-02 06:37:59.966886,2.0,15.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,01_00_call,NaN,0.0
1001201093_02_00_visit,10K_1001201093,2023-07-04 11:58:07.738595,NaN,15.0,NaN,NaN,NaN,NaN,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,02_00_visit,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9999226141_03_00_call,10K_9999226141,2023-03-19 09:04:11.612344,1.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,03_00_call,0.0,0.0
9999226141_04_00_visit,10K_9999226141,2024-01-15 16:20:17.350467,1.0,5.0,0.5,NaN,NaN,4.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,04_00_visit,0.0,0.0
9999226141_06_00_visit,10K_9999226141,2025-11-19 10:27:07.216672,1.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,06_00_visit,NaN,0.0


In [141]:
bottom_RF_female = pd.read_csv(DeepFolderPath + 'bottom_new_RF_scaled_female.csv', index_col=0)
top_RF_female = pd.read_csv(DeepFolderPath + 'top_new_RF_scaled_female.csv', index_col=0)
wavlm_filtered = pd.read_csv(DeepFolderPath + 'WavLM_filtered_w_details.csv', index_col=0)

In [147]:
# chnage the index of wavlm_filtered to be the first word before '_' in the current index added the visit_number column
wavlm_filtered['index'] = wavlm_filtered.index.to_series().apply(lambda x: f"{x.split('_')[0]}_{wavlm_filtered.loc[x, 'visit_number']}")
wavlm_filtered.set_index('index', inplace=True)

In [148]:
wavlm_filtered

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,age,gender,BMI,yob,month_of_birth,subject_number,visit_number,visit_date,days_from_today,accu_age
index,,,,,,,,,,,,,,,,,,,,,
3765205367_02_00_visit,-0.102321,-0.036604,-0.052482,0.012090,-0.105967,-0.009908,-0.091829,0.008895,-0.062261,-0.099200,...,43.0,1.0,33.601170,1981.0,8.0,3765205367,02_00_visit,2024-01-10,637,42.4
7481158686_02_00_visit,-0.046398,-0.067525,0.046671,0.019795,-0.150688,0.022957,-0.064874,0.004986,-0.069547,-0.118305,...,56.0,1.0,25.404949,1969.0,1.0,7481158686,02_00_visit,2025-04-23,168,56.3
1511328621_02_00_visit,-0.002198,-0.124653,0.017402,-0.026062,-0.087753,0.004575,-0.081135,0.034294,-0.006931,-0.088396,...,53.0,1.0,25.959978,1970.0,7.0,1511328621,02_00_visit,2023-12-12,666,53.4
2202638709_02_00_visit,-0.009686,-0.021650,0.011369,0.035600,-0.097757,0.022139,-0.079984,-0.000162,-0.076785,-0.102132,...,43.0,1.0,34.417561,1981.0,6.0,2202638709,02_00_visit,2024-09-19,384,43.3
3046237157_04_00_visit,0.026151,-0.045326,0.033846,-0.015478,-0.086539,-0.007088,-0.087950,-0.004284,-0.073187,-0.090341,...,45.0,1.0,27.377647,1979.0,6.0,3046237157,04_00_visit,2024-04-25,531,44.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8523928314_02_00_visit,-0.120390,0.054004,-0.008170,0.067010,-0.030316,-0.045063,-0.021244,-0.014228,-0.075605,0.025559,...,54.0,0.0,24.386526,1970.0,7.0,8523928314,02_00_visit,2024-03-06,581,53.6
3721132784_02_00_visit,-0.027016,-0.007186,-0.030357,0.040843,-0.072849,-0.006564,-0.073158,0.015149,-0.066784,-0.054663,...,69.0,0.0,25.191929,1955.0,8.0,3721132784,02_00_visit,2024-10-29,344,69.2
5033827218_04_00_visit,0.032200,-0.013765,0.066967,-0.022772,-0.091328,-0.001669,-0.080947,-0.000986,-0.112354,-0.106258,...,58.0,0.0,25.246700,1967.0,12.0,5033827218,04_00_visit,2025-07-14,86,57.6


In [149]:
wavlm_filtered_lifestyle = lifestyle_numbers_df.join(wavlm_filtered, how='inner')
bottom_female_lifestyle = lifestyle_numbers_df.join(bottom_RF_female, how='inner')
top_female_lifestyle = lifestyle_numbers_df.join(top_RF_female, how='inner')

In [152]:
# show the value counts of past_smoker and current_smoker in wavlm_filtered_lifestyle split by gender
# add percentages to the value counts
for gender_val, label in [(1, "male"), (0, "female")]:
    sub_df = wavlm_filtered_lifestyle[wavlm_filtered_lifestyle['gender'] == gender_val]
    print(f"\nGender: {label}")
    past_counts = sub_df['past_smoker'].value_counts(dropna=False)
    current_counts = sub_df['current_smoker'].value_counts(dropna=False)
    past_percentages = past_counts / len(sub_df) * 100
    current_percentages = current_counts / len(sub_df) * 100
    print("Past Smoker Counts:")
    for val, count in past_counts.items():
        percentage = past_percentages[val]
        print(f" - {val}: {count} ({percentage:.2f}%)")
    print("Current Smoker Counts:")
    for val, count in current_counts.items():
        percentage = current_percentages[val]
        print(f" - {val}: {count} ({percentage:.2f}%)")



Gender: male
Past Smoker Counts:
 - 0.0: 1496 (56.18%)
 - 1.0: 757 (28.43%)
 - nan: 410 (15.40%)
Current Smoker Counts:
 - 0.0: 2081 (78.14%)
 - nan: 314 (11.79%)
 - 1.0: 268 (10.06%)

Gender: female
Past Smoker Counts:
 - 0.0: 1694 (54.61%)
 - 1.0: 901 (29.05%)
 - nan: 507 (16.34%)
Current Smoker Counts:
 - 0.0: 2382 (76.79%)
 - nan: 404 (13.02%)
 - 1.0: 316 (10.19%)


In [161]:
bottom_female_lifestyle.current_smoker.value_counts(dropna=False)

current_smoker
0.0    526
NaN     96
1.0     79
Name: count, dtype: int64

In [164]:
from scipy.stats import chi2_contingency

# Get current smoker value counts for top and bottom females
top_current = top_female_lifestyle['current_smoker'].value_counts(dropna=False).sort_index()
bottom_current = bottom_female_lifestyle['current_smoker'].value_counts(dropna=False).sort_index()

print("=" * 60)
print("CHI-SQUARED TEST: Current Smoking Distribution in Top vs Bottom Females")
print("=" * 60)
print("\nTop Females - Current Smoking Distribution:")
print(top_current)
print(f"Total: {top_current.sum()}")
print(f"Percentages:\n{(top_current / top_current.sum() * 100).round(2)}")

print("\nBottom Females - Current Smoking Distribution:")
print(bottom_current)
print(f"Total: {bottom_current.sum()}")
print(f"Percentages:\n{(bottom_current / bottom_current.sum() * 100).round(2)}")

# Create contingency table
contingency_smoking = pd.DataFrame({
    'Top_Female': top_current,
    'Bottom_Female': bottom_current
}).fillna(0)

print("\nContingency Table:")
print(contingency_smoking)

# Perform chi-squared test
chi2_stat, p_value, dof, expected = chi2_contingency(contingency_smoking)

print("\n" + "=" * 60)
print("CHI-SQUARED TEST RESULTS:")
print("=" * 60)
print(f"Chi-squared statistic: {chi2_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"\nSignificance level: 0.05")
if p_value < 0.05:
    print(f"Result: SIGNIFICANT difference (p = {p_value:.4f} < 0.05)")
else:
    print(f"Result: NO significant difference (p = {p_value:.4f} >= 0.05)")

print("\nExpected frequencies:")
print(pd.DataFrame(expected, 
                   index=contingency_smoking.index, 
                   columns=contingency_smoking.columns).round(2))

CHI-SQUARED TEST: Current Smoking Distribution in Top vs Bottom Females

Top Females - Current Smoking Distribution:
current_smoker
0.0    523
1.0     70
NaN     79
Name: count, dtype: int64
Total: 672
Percentages:
current_smoker
0.0    77.83
1.0    10.42
NaN    11.76
Name: count, dtype: float64

Bottom Females - Current Smoking Distribution:
current_smoker
0.0    526
1.0     79
NaN     96
Name: count, dtype: int64
Total: 701
Percentages:
current_smoker
0.0    75.04
1.0    11.27
NaN    13.69
Name: count, dtype: float64

Contingency Table:
                Top_Female  Bottom_Female
current_smoker                           
0.0                    523            526
1.0                     70             79
NaN                     79             96

CHI-SQUARED TEST RESULTS:
Chi-squared statistic: 1.5918
P-value: 0.4512
Degrees of freedom: 2

Significance level: 0.05
Result: NO significant difference (p = 0.4512 >= 0.05)

Expected frequencies:
                Top_Female  Bottom_Female
curr

In [166]:
# Get past smoker value counts for top and bottom females
top_past = top_female_lifestyle['past_smoker'].value_counts(dropna=False).sort_index()
bottom_past = bottom_female_lifestyle['past_smoker'].value_counts(dropna=False).sort_index()

print("=" * 60)
print("CHI-SQUARED TEST: Past Smoking Distribution in Top vs Bottom Females")
print("=" * 60)
print("\nTop Females - Past Smoking Distribution:")
print(top_past)
print(f"Total: {top_past.sum()}")
print(f"Percentages:\n{(top_past / top_past.sum() * 100).round(2)}")

print("\nBottom Females - Past Smoking Distribution:")
print(bottom_past)
print(f"Total: {bottom_past.sum()}")
print(f"Percentages:\n{(bottom_past / bottom_past.sum() * 100).round(2)}")

# Create contingency table
contingency_past_smoking = pd.DataFrame({
    'Top_Female': top_past,
    'Bottom_Female': bottom_past
}).fillna(0)

print("\nContingency Table:")
print(contingency_past_smoking)

# Perform chi-squared test
chi2_stat_past, p_value_past, dof_past, expected_past = chi2_contingency(contingency_past_smoking)

print("\n" + "=" * 60)
print("CHI-SQUARED TEST RESULTS:")
print("=" * 60)
print(f"Chi-squared statistic: {chi2_stat_past:.4f}")
print(f"P-value: {p_value_past:.4f}")
print(f"Degrees of freedom: {dof_past}")
print(f"\nSignificance level: 0.05")
if p_value_past < 0.05:
    print(f"Result: SIGNIFICANT difference (p = {p_value_past:.4f} < 0.05)")
else:
    print(f"Result: NO significant difference (p = {p_value_past:.4f} >= 0.05)")

print("\nExpected frequencies:")
print(pd.DataFrame(expected_past, 
                   index=contingency_past_smoking.index, 
                   columns=contingency_past_smoking.columns).round(2))

CHI-SQUARED TEST: Past Smoking Distribution in Top vs Bottom Females

Top Females - Past Smoking Distribution:
past_smoker
0.0    360
1.0    205
NaN    107
Name: count, dtype: int64
Total: 672
Percentages:
past_smoker
0.0    53.57
1.0    30.51
NaN    15.92
Name: count, dtype: float64

Bottom Females - Past Smoking Distribution:
past_smoker
0.0    373
1.0    219
NaN    109
Name: count, dtype: int64
Total: 701
Percentages:
past_smoker
0.0    53.21
1.0    31.24
NaN    15.55
Name: count, dtype: float64

Contingency Table:
             Top_Female  Bottom_Female
past_smoker                           
0.0                 360            373
1.0                 205            219
NaN                 107            109

CHI-SQUARED TEST RESULTS:
Chi-squared statistic: 0.0989
P-value: 0.9518
Degrees of freedom: 2

Significance level: 0.05
Result: NO significant difference (p = 0.9518 >= 0.05)

Expected frequencies:
             Top_Female  Bottom_Female
past_smoker                           
0.0 

## predict from Voice for consistency

In [167]:
# get all cols in wavlm_filtered that contain 'feature' in their name
feature_cols = [col for col in wavlm_filtered.columns if 'feature' in col.lower()]

In [174]:
wavlm_filtered_male = wavlm_filtered[wavlm_filtered['gender'] == 1]
wavlm_filtered_female = wavlm_filtered[wavlm_filtered['gender'] == 0]

In [178]:
ridge_groupcv_with_exports(
    df=wavlm_filtered,
    target_col='age',
    group_col='subject_number',
    output_dir=DeepFolderPath + 'Ridge_stuff/voice_age_prediction',
    columns=feature_cols,
    handle_nans='impute',
    impute_strategy='median',
    n_splits=5,
    random_state=42,
    split_gender=True,
    alpha=0.1,
    standardize=False
    )

Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/voice_age_prediction/gender_male
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/voice_age_prediction/gender_female
Saved gender specific metrics in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/voice_age_prediction
  - metrics_by_gender.csv: tabular format
  - metrics_by_gender.json: structured format
  - gender_male/: male-specific outputs
  - gender_female/: female-specific outputs
Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/voice_age_prediction/gender_female
Saved gender specific metrics in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/Ridge_stuff/voice_age_prediction
  - metrics_by_gender.csv: tabular format
  - metrics_by_gender.json: structured format
  -

{'male': {'oof_Pearson_r': 0.6688961310582352,
  'oof_R2': 0.443429143510469,
  'oof_MAE': 4.362809905969281,
  'oof_RMSE': 5.533819400537852,
  'per_fold_R2': [0.450633250752985,
   0.4400666301839863,
   0.42852713781088125,
   0.4198168941674639,
   0.4770144613788553],
  'n_samples': 3764,
  'n_features': 1024,
  'alpha': 0.1,
  'alpha_optimized': False,
  'fold_best_alphas': None,
  'standardize': False,
  'handle_nans': 'impute',
  'impute_strategy': 'median',
  'split_gender_post_train': False,
  'regress_out_confounds': False,
  'confound_columns': None,
  'confound_alpha': None},
 'female': {'oof_Pearson_r': 0.7468738889841864,
  'oof_R2': 0.5561912337000604,
  'oof_MAE': 3.9039480641350406,
  'oof_RMSE': 4.913297668264678,
  'per_fold_R2': [0.5547823124184093,
   0.5678183133582977,
   0.5380911647524798,
   0.5804055557121832,
   0.5372091530944774],
  'n_samples': 4020,
  'n_features': 1024,
  'alpha': 0.1,
  'alpha_optimized': False,
  'fold_best_alphas': None,
  'standard

In [179]:
lightgbm_groupcv_with_exports(
    df=wavlm_filtered_male,
    target_col='age',
    group_col='subject_number',
    output_dir=DeepFolderPath + 'LGBM_stuff/voice_age_prediction',
    columns=feature_cols,
    handle_nans='impute',
    impute_strategy='median',
    n_splits=5,
    random_state=42,
    split_gender=False,
    optimize_hyperparams=False,
    )

Saved outputs in /net/mraid20/export/genie/LabData/Analyses/DeepVoiceFolder/Oct25_voice_full_length/LGBM_stuff/voice_age_prediction


{'oof_Pearson_r': 0.5274181762806097,
 'oof_R2': 0.27811030904539247,
 'oof_MAE': 5.089943476403401,
 'oof_RMSE': 6.302315935731861,
 'per_fold_R2': [0.3159152610821896,
  0.26125787362803965,
  0.26390845819510855,
  0.25868080122030124,
  0.28926897917744754],
 'n_samples': 3764,
 'n_features': 1024,
 'lgbm_params': {'objective': 'regression',
  'metric': 'rmse',
  'boosting_type': 'gbdt',
  'verbosity': -1,
  'seed': 42},
 'num_boost_round': 100,
 'handle_nans': 'impute',
 'impute_strategy': 'median',
 'split_gender_post_train': False,
 'regress_out_confounds': False,
 'confound_columns': None,
 'confound_alpha': None,
 'optimize_hyperparams': False,
 'fold_best_params': None}